# 03 — Train model SDC

Notebook này chứa toàn bộ hàm dùng chung cho dữ liệu, train và suy luận. `01`, `02`, `04`, `05`, `06` nạp các định nghĩa ở đây bằng `%run -i` với cờ `SDC_IMPORT_ONLY`, nên **không chạy train** khi nạp.

Chạy `Run All` trực tiếp ở đây để tạo candidate model. Sau khi sửa hàm dùng chung, khởi động lại kernel của notebook gọi để nạp bản mới.


In [1]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("Configured for 88-core CPU")

Configured for 88-core CPU


## 1. Nền: đường dẫn, lược đồ, mã hoá, L1 và semantic


In [2]:
# Xác định gốc dự án từ thư mục làm việc của kernel Jupyter.
from pathlib import Path

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "Code").is_dir() and (p / "Data").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Hãy mở notebook từ bên trong dự án SDC")
CODE_DIR = ROOT / "Code"
print("Gốc dự án:", ROOT)

Gốc dự án: /home/ubuntu/sepcung/02.SDC


### Đường dẫn dự án

Duong dan co dinh cua du an - mot dinh nghia duy nhat cho moi module va notebook.

Tach rieng vi day la thu duy nhat phu thuoc vi tri file tren dia. Notebook khong co
`__file__`, nen ban notebook thu vien thay dong `ROOT` nay bang phep do thu muc.

In [3]:
from pathlib import Path


DATA = ROOT / "Data"
FEAT_DIR = DATA / "features"
MODELS = ROOT / "Models"
REPORTS = ROOT / "Reports"
SESSIONS_PATH = DATA / "sessions_verified.parquet"

# Bảng thu nạp nằm ở `Models/`, KHÔNG nằm trong `Models/<run_id>/`. Thu nạp là kiến thức
# của người vận hành về thiết bị ngoài hiện trường, không thuộc về một lần train nào —
# để nó trong thư mục run thì mỗi lần train lại là mất sạch, mà thu nạp lại được sinh ra
# đúng vì không muốn phải train lại.
ENROLLED_PATH = MODELS / "enrolled.json"

### Lược đồ dữ liệu — head, nhóm feature, chia tập

Luoc do du lieu: ten head, nhom cot feature, va cach chia tap.

Chi mo ta DU LIEU trong ra sao - khong ma hoa, khong tra bang, khong train. Moi module
khac hoi file nay "cot nao la nhan, cot nao la feature, feature nay tu nguon nao".

In [4]:
import pandas as pd


MISSING = "<missing>"

META_COLS = ["canonical_device", "scenario", "capture_file", "date", "mac", "n_sources"]

# Ba head, đúng thứ tự output của sản phẩm: hãng / loại / model sản phẩm.
#
# Head thứ ba từng tên là `family` (contract ≤ 2.1). Đổi thành `model` cho khớp ô MODEL
# trong spec, nhưng chỉ là ĐỔI TÊN — nội dung nhãn không đổi, và độ mịn của nó vẫn là
# "model sản phẩm ở mức phân biệt được bằng telemetry": `Amazon Echo Dot` là một model,
# còn `Tuya Plug` vẫn gộp 13 thiết bị ODM không tách được bằng DHCP/DNS/TLS. Đổi tên
# không làm chúng tách ra được; xem `Data/device_labels_verified.csv` cột review_note.
MODEL_HEAD = "model"
LEGACY_MODEL_HEAD = "family"
HEAD_ALIASES = {LEGACY_MODEL_HEAD: MODEL_HEAD}
LABEL_COLS = ["make", "type", MODEL_HEAD]
SOURCES = ["dhcp", "dns", "mdns", "tls"]

# Cột dạng chuỗi token -> TF-IDF. Phần còn lại dạng chuỗi -> categorical nguyên giá trị.
TEXT_COLS = ["dhcp_vci", "dns_tokens", "mdns_tokens", "tls_sni_tokens"]

# Feature biến mất khi DNS được mã hoá (DoH/DoT) và SNI bị ẩn (ECH). Đây là lý do thật
# để đo bản không có chúng — không phải vì sợ leak: lúc chạy thật router vẫn đọc được
# DNS query và SNI, nên chúng là bằng chứng hợp lệ.
ENCRYPTED_RISK = ["dns_tokens", "tls_sni_tokens"]

# Field driver KHÔNG cung cấp — không feature nào được suy ra từ chúng
FORBIDDEN = ["src_ip", "dst_ip", "dst_port", "ts", "iat", "pktsize", "bytes", "rate"]


# Cot co-mat cua tung nguon. Suy tu `SOURCES` de danh sach nguon chi co MOT ban.
SOURCE_FLAGS = tuple(f"has_{source}" for source in SOURCES)

RARE_THRESHOLD = 10      # lớp dưới ngưỡng này không học được, tách riêng khi report


def source_of(col):
    """Cột feature này suy từ nguồn nào. `has_tls` -> 'tls', `dhcp_opt_3` -> 'dhcp'."""
    return col[4:] if col.startswith("has_") else col.split("_", 1)[0]


def feature_sets(df):
    """Các nhóm feature, suy từ chính các cột của dataframe."""
    feats = [c for c in df.columns if c not in META_COLS + LABEL_COLS]
    text = [c for c in TEXT_COLS if c in feats]
    cat = [c for c in feats if c not in text and not pd.api.types.is_numeric_dtype(df[c])]
    num = [c for c in feats if c not in text and c not in cat]
    return {
        "all": feats,
        "text": text,
        "cat": cat,
        "num": num,
        "source": {s: [c for c in feats if c.startswith(s)] for s in SOURCES},
        "no_content": [c for c in feats if c not in ENCRYPTED_RISK],
    }


def assert_serveable(cols):
    """Fail nếu có feature vượt quá telemetry driver cung cấp."""
    bad = [c for c in cols if any(f in c for f in FORBIDDEN)]
    assert not bad, f"Feature không phục vụ được lúc chạy thật: {bad}"
    orphan = [c for c in cols if not c.startswith(("dhcp_", "dns_", "mdns_", "tls_", "has_"))]
    assert not orphan, f"Feature không truy được về nguồn nào: {orphan}"
    return True


def normalize_heads(df):
    """Đổi tên cột nhãn cũ sang tên head hiện hành (`family` -> `model`).

    Làm ở đây, một chỗ, thay vì ghi lại parquet: notebook 02 và các file dữ liệu đã sinh
    ra từ trước vẫn dùng được, mà phần còn lại của code chỉ biết đúng một tên. Cột mới
    đã có sẵn thì không đụng vào — dữ liệu mới luôn thắng.
    """
    rename = {old: new for old, new in HEAD_ALIASES.items()
              if old in df.columns and new not in df.columns}
    # Có cả hai cột: cột mới là nhãn đang dùng, cột cũ là tàn dư của file dựng trước lần
    # đổi tên. Bỏ hẳn cột cũ — để lại thì nó không phải nhãn mà cũng không phải feature,
    # và `assert_serveable` sẽ bắt nó như một feature không truy được về nguồn nào.
    stale = [old for old, new in HEAD_ALIASES.items()
             if old in df.columns and new in df.columns]
    if rename or stale:
        df = df.rename(columns=rename).drop(columns=stale)
    return df


def load_sessions(path=SESSIONS_PATH):
    """Nạp feature table mức session kèm nhóm feature đã kiểm tra."""
    assert path.exists(), f"Không tìm thấy {path} — chạy 02_build_dataset.ipynb trước"
    df = normalize_heads(pd.read_parquet(path))
    fs = feature_sets(df)
    assert_serveable(fs["all"])
    return df, fs


# --- Chia tập ----------------------------------------------------------------------

def date_groups(df):
    """Group key cho bài 'thiết bị đã biết, thời điểm khác'. Dùng ngày capture chứ không
    dùng ts: capture IDLE kéo qua nửa đêm nên suy từ ts sẽ xé một capture thành 2 group."""
    return df["date"].to_numpy()


def device_groups(df):
    """Group key cho bài 'thiết bị chưa từng thấy' (LOGO ở notebook 04)."""
    return df["canonical_device"].to_numpy()


def rare_classes(y, threshold=RARE_THRESHOLD):
    vc = pd.Series(y).value_counts()
    return set(vc[vc < threshold].index)

### Bộ phân lớp L2

Dinh nghia bo phan lop L2 - dung chung cho train va moi lan danh gia.

Sieu tham so nam canh ham dung model: nguong abstain hieu chinh cho model 250 cay khong
dung voi model 25 cay, nen hai thu nay khong duoc phep lech nhau.

In [5]:
from sklearn.ensemble import RandomForestClassifier

# Số cây của RandomForest. Một định nghĩa cho cả 03 (train), 04 (hiệu chỉnh ngưỡng) và
# 05 (xuất ONNX) — ba chỗ này BẮT BUỘC dùng chung một giá trị, vì ngưỡng abstain hiệu
# chỉnh cho model 250 cây không đúng với model 25 cây.
#
# Đo được (CV 5 fold theo ngày): 25 / 100 / 250 cây cho macro-F1 như nhau trong biên độ
# nhiễu — `type` lần lượt 0.8442 / 0.8443 / 0.8440, và 50 cây còn ra 0.8331, tức không
# đơn điệu. Kích thước thì khác hẳn: 1.54 / 6.07 / 15.80 MB.
#
# ⚠️ Số cây ảnh hưởng tới ĐỘ PHÂN GIẢI của confidence mạnh hơn tới accuracy: 25 cây chỉ
# sinh được 26 mức xác suất (bội của 1/25), nên đường coverage–accuracy thành bậc thang
# thô và ngưỡng chọn được kém tinh. Với kiến trúc này, confidence là thứ quyết định phát
# hiện thiết bị lạ, nên đổi số cây thì phải đo lại `ood_abstain` chứ đừng chỉ nhìn F1.
N_ESTIMATORS = 250

RNG = 42


def make_model() -> RandomForestClassifier:
    """Bo phan lop chuan cua du an. Train va danh gia BAT BUOC goi cung ham nay."""
    return RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RNG,
    )

### Bộ mã hoá feature (TF-IDF + ordinal)

Bo ma hoa feature (TF-IDF + ordinal) duoi dang mot `ColumnTransformer`.

Vocab sinh ra luc train phai giong het luc export model contract; lech thi model chay
tren thiet bi decode sai trong khi test offline van xanh. Vi vay `fit_encoder` /
`apply_encoder` chi duoc dinh nghia o day.

In [6]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OrdinalEncoder


# Số chiều TF-IDF mỗi cột text. Cộng lại ra kích thước input tensor nên phải cố định
# và ghi vào model contract.
TFIDF_MAX_FEATURES = {"dns_tokens": 200, "mdns_tokens": 100,
                      "tls_sni_tokens": 150, "dhcp_vci": 100}

# --- Encode -------------------------------------------------------------------------
# Bộ mã hoá là một `ColumnTransformer` của sklearn, không phải code tự ghép. Lý do không
# phải vì gọn hơn, mà vì **xuất được sang ONNX**: skl2onnx chuyển được cả cụm
# ColumnTransformer + TF-IDF + RandomForest thành MỘT graph nhận chuỗi thô, nên user space
# không phải cài lại TF-IDF — nguồn train/serve skew nguy hiểm nhất của pipeline này.
#
# Hai quy ước đổi so với contract 1.x, đều vì ràng buộc của skl2onnx:
#   - `dhcp_vci` dùng analyzer 'word' tách theo ':' '-' '.' '_' thay vì char_wb 3-5gram.
#     skl2onnx chỉ chuyển được analyzer='word'. Đo được (CV 5 fold theo ngày): cách mới
#     KHÔNG tệ hơn — macro-F1 `type` 0.8331 -> 0.8440, make/family không đổi. Tách
#     'dhcpcd-6.8.2:Linux-4.4.22+:armv7l:MT8167B' theo dấu phân cách cho ra token có
#     nghĩa (dhcpcd, Linux, armv7l, MT8167B), hoá ra hợp với trường này hơn char n-gram.
#   - Chỉ số OOV là -1, không phải len(vocab). Đây là quy ước của `OrdinalEncoder`.
#
# Cả hai đổi đều phá contract cũ nên `contract_version` phải là 2.x.

VCI_TOKEN_PATTERN = r"[^:\-_. ]+"   # dhcp_vci: tách theo dấu phân cách trong chính chuỗi
TOKEN_PATTERN = r"[^ ]+"            # còn lại: bước 02 đã tách token sẵn, chỉ cắt khoảng trắng
OOV_INDEX = -1                      # giá trị categorical chưa từng thấy


def _make_vectorizer(col):
    pattern = VCI_TOKEN_PATTERN if col == "dhcp_vci" else TOKEN_PATTERN
    return TfidfVectorizer(analyzer="word", token_pattern=pattern,
                           max_features=TFIDF_MAX_FEATURES[col], min_df=2)


def build_encoder(num, cat, text):
    """ColumnTransformer chuẩn của project. Thứ tự khối: numeric, categorical, rồi text."""
    steps = []
    if num:
        steps.append(("num", "passthrough", list(num)))
    if cat:
        steps.append(("cat", OrdinalEncoder(handle_unknown="use_encoded_value",
                                            unknown_value=OOV_INDEX,
                                            encoded_missing_value=OOV_INDEX), list(cat)))
    for c in text:
        steps.append((f"t_{c}", _make_vectorizer(c), c))   # chuỗi, không phải list
    return ColumnTransformer(steps, sparse_threshold=0)


def prepare_frame(df, encoder):
    """Ép kiểu đúng như lúc fit: numeric float32, categorical/text chuỗi.

    ColumnTransformer khớp cột theo TÊN nên thứ tự cột đầu vào không quan trọng, nhưng
    kiểu thì có: TF-IDF nhận object/str, để lọt NaN vào là vỡ.
    """
    num, cat, text = encoder["num_cols"], encoder["cat_cols"], encoder["text_cols"]
    out = df[num + cat + text].copy()
    for c in num:
        out[c] = out[c].astype(np.float32)
    for c in cat + text:
        out[c] = out[c].astype(str)
    return out


def fit_encoder(df, cols, min_count=1):
    """Trả dict mô tả toàn bộ phép encode — chính là phần `vocab` của model contract."""
    fs = feature_sets(df)
    num = [c for c in fs["num"] if c in cols]
    cat = [c for c in fs["cat"] if c in cols]
    text = [c for c in fs["text"] if c in cols]
    encoder = {"num_cols": num, "cat_cols": cat, "text_cols": text,
               "ct": build_encoder(num, cat, text)}
    encoder["ct"].fit(prepare_frame(df, encoder))
    return encoder


def encoder_vectorizers(encoder):
    """TF-IDF đã fit, theo cột — 05 cần để ghi tham số vào contract."""
    named = encoder["ct"].named_transformers_
    return {c: named[f"t_{c}"] for c in encoder["text_cols"]}


def encoder_categories(encoder):
    """Vocab categorical đã đóng băng, theo cột."""
    if not encoder["cat_cols"]:
        return {}
    enc = encoder["ct"].named_transformers_["cat"]
    return {c: [str(v) for v in cats]
            for c, cats in zip(encoder["cat_cols"], enc.categories_)}


def apply_encoder(df, encoder):
    """Trả (X, feature_names, cat_index). cat_index dùng cho HistGradientBoosting."""
    num, cat, text = encoder["num_cols"], encoder["cat_cols"], encoder["text_cols"]
    X = encoder["ct"].transform(prepare_frame(df, encoder)).astype(np.float32)
    names = list(num) + list(cat)
    vecs = encoder_vectorizers(encoder)
    for c in text:
        names += [f"{c}::{t}" for t in vecs[c].get_feature_names_out()]
    cat_index = list(range(len(num), len(num) + len(cat)))
    return X, names, cat_index

### Tầng L1 — bảng vân tay và sổ thu nạp

Tang L1: bang tra van tay exact-match, va so thu nap cua nguoi van hanh.

`MODES` duoc ghi thang vao model.joblib va duong chay that duyet lai dung thu tu do,
nen sua o day la sua cho ca luc train lan luc chay.

In [7]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd


# --- Tầng L1 ------------------------------------------------------------------------
# Đặt ở đây vì cả 03 (dựng + đo), 04 (dựng lại theo từng fold LOGO) và 05 (đóng gói vào
# contract) đều cần đúng một định nghĩa. `MODES` được ghi thẳng vào model.joblib và
# `sdc_infer.l1_lookup` duyệt lại đúng thứ tự đó, nên sửa ở đây là sửa cho cả đường chạy thật.

In [8]:
class FingerprintTable:
    """Tầng L1 — tra bảng exact match, từ khoá cụ thể nhất tới chung nhất.

    `min_support` đếm session, `min_devices` đếm **thiết bị khác nhau** cùng mang khoá đó.
    Hai cái không thay thế nhau được: 70 session của đúng một thiết bị trông rất chắc
    nhưng vẫn chỉ là bằng chứng của một thiết bị. Vân tay TLS hay gặp nhất là vân tay của
    *thư viện TLS* chứ không phải của hãng — `iRobot Roomba` và `Ring Base Station` dùng
    chung `0303|4866,4867,4865,...` (cipher suite TLS 1.3 chuẩn) dù không liên quan gì
    nhau. Chỉ một thiết bị chống lưng thì không phân biệt được "vân tay của hãng này" với
    "vân tay của thư viện này", và khi thiết bị kia xuất hiện, bảng trả lời sai với
    confidence 1.0 — lỗi im lặng, không tầng nào chặn.

    Đo ở notebook 04: `min_devices=2` đưa tỉ lệ lỗi im lặng trên thiết bị lạ từ 3.4% về 0
    cho `make` và từ 2.0% về 0 cho `family`, đổi lại L1 coverage giảm 76.5% -> 33.5%. Phần
    mất đi rơi xuống L2 (accuracy 0.998) và **chịu ngưỡng abstain**, nên đây là đánh đổi
    có lợi cho đường thu nạp thiết bị mới.

    ⚠️ Gác này chỉ áp cho khoá **đào từ dữ liệu train** (`tables`). Khoá do người vận hành
    thêm tay khi thu nạp một thiết bị mới chỉ có đúng một thiết bị chống lưng, và nó đáng
    tin vì người khẳng định chứ không vì số liệu — nên nó đi vào `enrolled` qua `enroll()`,
    không qua `fit()`.

    Ba sổ, tra theo đúng thứ tự này:

    | Sổ | Nguồn | Ý nghĩa khi khớp |
    |---|---|---|
    | `enrolled` | người vận hành | trả lời, `source="enrolled"` |
    | `tables` | đào từ train, đã qua gác | trả lời, `source="mined"` |
    | `ambiguous` | khoá đào được nhưng ứng **nhiều nhãn** | `status="ambiguous"` |

    Sổ nhập nhằng là chỗ trước đây bị vứt đi, và vứt nó là một lỗi thiết kế. Đo ở
    notebook 04: 13 thiết bị cụm Tuya dùng chung một vân tay nhưng mang 3 `type` khác
    nhau, nên chúng bị abstain — và ở mức MAC, abstain bị đọc nhầm thành "thiết bị lạ",
    làm 14/40 thiết bị đã biết bị báo động nhầm ở head `type`. Giữ khoá lại kèm tập nhãn
    cho phép phân biệt "chưa từng thấy vân tay này" (thu nạp được) với "biết vân tay
    nhưng nó không tách được loại" (thu nạp vô ích, phải thêm nguồn bằng chứng khác).
    """

    MODES = [
        ("fp_full", ["dhcp_prl", "dhcp_vci", "tls_fp"], ["has_dhcp", "has_tls"]),
        ("fp_dhcp", ["dhcp_prl", "dhcp_vci"], ["has_dhcp"]),
        ("fp_tls", ["tls_fp"], ["has_tls"]),
    ]

    DEVICE_COL = "canonical_device"
    KEY_SEP = " || "

    def __init__(self, min_support=3, min_devices=2):
        self.min_support = min_support
        self.min_devices = min_devices
        self.tables = {}        # (mode, head) -> {key: label}
        self.ambiguous = {}     # (mode, head) -> {key: [label, ...]}
        self.enrolled = {}      # (mode, head) -> {key: label}

    @classmethod
    def _key(cls, frame, cols):
        return frame[cols].astype(str).agg(cls.KEY_SEP.join, axis=1)

    @classmethod
    def _key_of_row(cls, row, cols):
        return cls.KEY_SEP.join(str(row[c]) for c in cols)

    @staticmethod
    def _available(frame, flags):
        return (frame[flags] == 1).all(axis=1)

    @classmethod
    def _usable(cls, frame, cols, flags):
        """Rows whose source is present and whose entire fingerprint key is usable."""
        available = cls._available(frame, flags)
        values = frame[cols]
        present = values.notna().all(axis=1)
        for col in cols:
            present &= values[col].astype(str).str.strip().ne("")
            present &= values[col].astype(str).ne(MISSING)
        return available & present

    @staticmethod
    def _value_present(value):
        if value is None:
            return False
        try:
            if pd.isna(value):
                return False
        except (TypeError, ValueError):
            pass
        return str(value).strip() not in ("", MISSING)

    def fit(self, train, heads):
        for mode, cols, flags in self.MODES:
            sub = train[self._usable(train, cols, flags)]
            if sub.empty:
                self.tables.update({(mode, h): {} for h in heads})
                self.ambiguous.update({(mode, h): {} for h in heads})
                continue
            keys = self._key(sub, cols)
            for head in heads:
                stats = (pd.DataFrame({"k": keys.to_numpy(), "y": sub[head].to_numpy(),
                                       "d": sub[self.DEVICE_COL].to_numpy()})
                         .groupby("k").agg(n_label=("y", "nunique"), n=("y", "count"),
                                           n_dev=("d", "nunique"), label=("y", "first"),
                                           labels=("y", lambda s: sorted(set(s)))))
                enough = stats.n >= self.min_support
                pure = stats[(stats.n_label == 1) & enough & (stats.n_dev >= self.min_devices)]
                self.tables[(mode, head)] = pure.label.to_dict()
                # Khoá nhiều nhãn: giữ lại thay vì vứt, để phân biệt 'chưa từng thấy'
                # với 'thấy rồi nhưng vân tay không tách được nhãn'.
                self.ambiguous[(mode, head)] = stats.loc[(stats.n_label > 1) & enough,
                                                         "labels"].to_dict()
        return self

    def predict(self, test, head):
        """Tra cả một bảng (dùng ở notebook 03/04). Trả (nhãn, mode đã dùng).

        Khoá nhập nhằng tính là `miss`: ở đây L1 không đưa ra nhãn nào, đúng như lúc
        chạy thật. Phân biệt `ambiguous` với `miss` chỉ có nghĩa ở tầng phục vụ (quyết
        định có thu nạp hay không) nên nó nằm ở `lookup()`, không nằm ở đây.
        """
        pred = pd.Series(None, index=test.index, dtype=object)
        used = pd.Series("miss", index=test.index, dtype=object)
        for mode, cols, flags in self.MODES:
            todo = pred.isna() & self._usable(test, cols, flags)
            if not todo.any():
                continue
            keys = self._key(test[todo], cols)
            book = {**self.tables.get((mode, head), {}),
                    **self.enrolled.get((mode, head), {})}
            hit = keys.map(book)
            found = hit[hit.notna()]
            pred.loc[found.index] = found
            used.loc[found.index] = mode
        return pred, used

    # --- Tra một dòng (đường chạy thật) ---------------------------------------------

    def best_mode(self, row):
        """Mode cụ thể nhất mà bằng chứng của `row` đủ để dựng khoá. None nếu không có."""
        for mode, cols, flags in self.MODES:
            if (all(row.get(f) == 1 for f in flags)
                    and all(self._value_present(row.get(c)) for c in cols)):
                return mode, cols
        return None, None

    def lookup(self, row, head):
        """Tra một dòng feature.

        Trả dict với `status` thuộc {'hit', 'ambiguous', 'miss'}. Ba trạng thái này là
        thứ tầng trên dùng để quyết định có thu nạp hay không — gộp 'ambiguous' vào
        'miss' sẽ dẫn tới thu nạp những vân tay vốn không tách được nhãn.
        """
        source_available = False
        for mode, cols, flags in self.MODES:
            if not all(row.get(f) == 1 for f in flags):
                continue
            source_available = True
            if not all(self._value_present(row.get(c)) for c in cols):
                continue
            key = self._key_of_row(row, cols)
            for source, book in (("enrolled", self.enrolled), ("mined", self.tables)):
                hit = book.get((mode, head), {}).get(key)
                if hit is not None:
                    return {"status": "hit", "label": hit, "labels": [hit],
                            "mode": mode, "source": source}
            amb = self.ambiguous.get((mode, head), {}).get(key)
            if amb is not None:
                # Mode cụ thể hơn đã nhập nhằng thì mode chung hơn không thể khá hơn.
                return {"status": "ambiguous", "label": None, "labels": list(amb),
                        "mode": mode, "source": "mined"}
        status = "insufficient_key" if source_available and self.best_mode(row)[0] is None else "miss"
        return {"status": status, "label": None, "labels": [], "mode": "miss",
                "source": None}

    # --- Thu nạp ---------------------------------------------------------------------

    def enroll(self, row, head, label):
        """Thêm khoá do người vận hành xác nhận. Không qua gác `min_devices`.

        Ghi ở mode **cụ thể nhất** mà bằng chứng cho phép. Thu nạp chỉ bằng `fp_tls` được
        đánh dấu `provisional`: vân tay TLS đơn lẻ là vân tay của thư viện TLS chứ chưa
        chắc của hãng (xem docstring lớp), nên nó cần được nâng cấp lên `fp_full` khi MAC
        đó lộ DHCP.

        Nếu khoá đã ứng với một nhãn khác thì đây chính là ca va chạm vân tay: khoá bị
        chuyển sang sổ nhập nhằng và **không sổ nào trả lời nữa** — đúng hơn là để một
        trong hai nhãn thắng tuỳ thứ tự thu nạp.
        """
        mode, cols = self.best_mode(row)
        if mode is None:
            return {"action": "no_fingerprint", "mode": None}
        key = self._key_of_row(row, cols)
        provisional = mode == "fp_tls"

        amb = self.ambiguous.get((mode, head), {}).get(key)
        if amb is not None:
            if label not in amb:
                amb.append(label)
                amb.sort()
            return {"action": "still_ambiguous", "mode": mode, "labels": list(amb)}

        for book in (self.enrolled, self.tables):
            existing = book.get((mode, head), {}).get(key)
            if existing is not None and existing != label:
                labels = sorted({existing, label})
                self.ambiguous.setdefault((mode, head), {})[key] = labels
                self.enrolled.get((mode, head), {}).pop(key, None)
                self.tables.get((mode, head), {}).pop(key, None)
                return {"action": "conflict", "mode": mode, "labels": labels}
            if existing == label:
                return {"action": "already_known", "mode": mode, "labels": [label]}

        self.enrolled.setdefault((mode, head), {})[key] = label
        return {"action": "enrolled", "mode": mode, "labels": [label],
                "provisional": provisional}

    def enrolled_doc(self):
        """Bảng thu nạp dạng JSON-được: {mode: {head: {key: label}}}."""
        doc = {}
        for (mode, head), book in sorted(self.enrolled.items()):
            if book:
                doc.setdefault(mode, {})[head] = dict(book)
        return doc

    def load_enrolled(self, doc):
        for mode, heads in (doc or {}).items():
            for head, book in heads.items():
                self.enrolled.setdefault((mode, head), {}).update(book)
        return self

In [9]:
def read_enrolled(path=ENROLLED_PATH):
    """Bảng thu nạp dùng chung cho mọi run. Chưa có file -> dict rỗng, không phải lỗi."""
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8")).get("enrolled", {})


def write_enrolled(table, path=ENROLLED_PATH, note=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    doc = {
        "format": "sdc-enrolled-v1",
        "updated": datetime.now().isoformat(timespec="seconds"),
        "note": note or "Khoá do người vận hành xác nhận; không đi qua gác min_devices.",
        "enrolled": table.enrolled_doc(),
    }
    path.write_text(json.dumps(doc, indent=2, ensure_ascii=False), encoding="utf-8")
    return path

### Catalog `type` mở theo ngữ nghĩa

Open-vocabulary device type inference from observable protocol evidence.

The catalog is deliberately outside the model artifact.  Adding a new protocol
signature or functional type does not require retraining the closed-set classifier.
Rules may only inspect telemetry supplied by the driver; identity metadata such as
MAC, IP, capture filename and evaluation device name is never included.

In [10]:
import json
import math
from pathlib import Path
import re



DEFAULT_CATALOG = DATA / "device_type_rules.json"
FORMAT = "sdc-semantic-type-v1"
FIELDS = ("dhcp", "dns", "mdns", "tls", "any")
ROW_FIELDS = {
    "dhcp": ("dhcp_vci", "dhcp_hostname"),
    "dns": ("dns_tokens",),
    "mdns": ("mdns_tokens",),
    "tls": ("tls_sni_tokens", "tls_alpn"),
}
RECORD_FIELDS = {
    "dhcp": ("hostname", "vendor_class_id", "opt60"),
    "dns": ("qname", "qry_name"),
    "mdns": ("qname", "qry_name", "service", "instance"),
    "tls": ("sni", "alpn"),
}


class SemanticCatalogError(ValueError):
    pass


def _clean(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    text = str(value).strip().lower()
    return "" if text == "<missing>" else text


def collect_evidence(row=None, records=None) -> dict[str, str]:
    """Collect only serveable semantic text, grouped by protocol source."""
    values = {field: [] for field in FIELDS if field != "any"}
    row = row or {}
    for source, columns in ROW_FIELDS.items():
        values[source].extend(filter(None, (_clean(row.get(column)) for column in columns)))
    for record in records or []:
        if not isinstance(record, dict):
            continue
        source = str(record.get("proto", "")).lower()
        if source not in RECORD_FIELDS:
            continue
        values[source].extend(
            filter(None, (_clean(record.get(column)) for column in RECORD_FIELDS[source]))
        )
    evidence = {source: " ".join(parts) for source, parts in values.items()}
    evidence["any"] = " ".join(evidence[source] for source in values)
    return evidence


class SemanticTypeClassifier:
    def __init__(self, catalog=DEFAULT_CATALOG):
        if isinstance(catalog, (str, Path)):
            path = Path(catalog)
            try:
                document = json.loads(path.read_text(encoding="utf-8"))
            except (OSError, UnicodeError, json.JSONDecodeError) as exc:
                raise SemanticCatalogError(f"cannot read semantic catalog {path}: {exc}") from exc
            self.catalog_path = path
        elif isinstance(catalog, dict):
            document = catalog
            self.catalog_path = None
        else:
            raise SemanticCatalogError("catalog must be a path or object")
        self._load(document)

    def _load(self, document):
        if not isinstance(document, dict) or document.get("format") != FORMAT:
            raise SemanticCatalogError(f"semantic catalog format must be {FORMAT!r}")
        self.catalog_version = str(document.get("catalog_version", "unversioned"))
        self.min_score = self._positive(document.get("min_score"), "min_score")
        self.ambiguity_margin = self._positive(
            document.get("ambiguity_margin"), "ambiguity_margin", allow_zero=True
        )
        raw_rules = document.get("rules")
        if not isinstance(raw_rules, list) or not raw_rules:
            raise SemanticCatalogError("semantic catalog rules must be a non-empty list")
        self.rules = []
        self.coverage = {}
        labels = set()
        for index, raw in enumerate(raw_rules):
            if not isinstance(raw, dict) or not str(raw.get("type", "")).strip():
                raise SemanticCatalogError(f"rule {index} needs a non-empty type")
            label = str(raw["type"]).strip()
            if label in labels:
                raise SemanticCatalogError(f"duplicate semantic type: {label}")
            labels.add(label)
            covers = raw.get("covers", [])
            if (not isinstance(covers, list)
                    or any(not isinstance(item, str) or not item.strip() for item in covers)):
                raise SemanticCatalogError(f"rule {label!r} covers must be a list of labels")
            self.coverage[label.casefold()] = {
                label.casefold(), *(item.strip().casefold() for item in covers)
            }
            signals = raw.get("signals")
            if not isinstance(signals, list) or not signals:
                raise SemanticCatalogError(f"rule {label!r} needs signals")
            compiled = []
            for signal in signals:
                field = signal.get("field") if isinstance(signal, dict) else None
                if field not in FIELDS:
                    raise SemanticCatalogError(f"invalid field in rule {label!r}: {field!r}")
                weight = self._positive(signal.get("weight"), f"{label}.weight")
                try:
                    pattern = re.compile(str(signal.get("pattern", "")), re.IGNORECASE)
                except re.error as exc:
                    raise SemanticCatalogError(f"invalid regex in rule {label!r}: {exc}") from exc
                if not pattern.pattern:
                    raise SemanticCatalogError(f"empty regex in rule {label!r}")
                compiled.append((field, pattern, weight))
            self.rules.append((label, compiled))

    def is_compatible(self, predicted, truth) -> bool:
        """Whether an open broad label is an accepted parent of the ground-truth type."""
        if not predicted or not truth:
            return False
        return str(truth).strip().casefold() in self.coverage.get(
            str(predicted).strip().casefold(), {str(predicted).strip().casefold()}
        )

    @staticmethod
    def _positive(value, name, allow_zero=False):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise SemanticCatalogError(f"{name} must be numeric")
        value = float(value)
        if not math.isfinite(value) or value < 0 or (not allow_zero and value == 0):
            raise SemanticCatalogError(f"{name} must be {'non-negative' if allow_zero else 'positive'}")
        return value

    def classify(self, row=None, records=None) -> dict:
        evidence = collect_evidence(row, records)
        candidates = []
        for label, signals in self.rules:
            score = 0.0
            matches = []
            for field, pattern, weight in signals:
                match = pattern.search(evidence[field])
                if match:
                    score += weight
                    matches.append({
                        "field": field,
                        "match": match.group(0),
                        "weight": weight,
                    })
            if score:
                candidates.append({"type": label, "score": score, "evidence": matches})
        candidates.sort(key=lambda item: (-item["score"], item["type"]))
        if not candidates:
            return {"status": "abstain", "type": None, "score": 0.0,
                    "reason": "no_semantic_evidence", "candidates": []}
        best = candidates[0]
        second_score = candidates[1]["score"] if len(candidates) > 1 else 0.0
        margin = best["score"] - second_score
        if best["score"] < self.min_score:
            reason = "semantic_score_below_threshold"
            status = "abstain"
        elif margin < self.ambiguity_margin:
            reason = "semantic_evidence_ambiguous"
            status = "abstain"
        else:
            reason = "semantic_type_evidence"
            status = "answer"
        confidence = best["score"] / (best["score"] + self.min_score)
        return {
            "status": status,
            "type": best["type"] if status == "answer" else None,
            "candidate_type": best["type"],
            "score": best["score"],
            "margin_score": margin,
            "confidence": round(float(confidence), 4),
            "threshold": 0.5,
            "reason": reason,
            "catalog_version": self.catalog_version,
            "evidence": best["evidence"],
            "candidates": [{"type": item["type"], "score": item["score"]}
                           for item in candidates[:3]],
        }

## 2. Dữ liệu: pcap, bản ghi raw và session


### Trích xuất feature từ pcap và bản ghi raw

Shared SDC feature extraction for training, evaluation, and serving.

The public pipeline is:

    packets -> extract_records()/read_pcap() -> aggregate() -> model feature row

Packet parsing is optional at import time so production callers that already receive
raw DHCP/DNS/mDNS/TLS records do not need Scapy installed.

In [11]:
import hashlib
import re
import struct
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd


try:  # Packet parsing is not needed by record-only production deployments.
    from scapy.all import BOOTP, DHCP, DNS, Ether, IP, IPv6, PcapReader, Raw, TCP, UDP
except ImportError:  # pragma: no cover - exercised only in minimal production images
    BOOTP = DHCP = DNS = Ether = IP = IPv6 = PcapReader = Raw = TCP = UDP = None


MAX_TOKENS = 40
MAX_DNS_PER_WINDOW = 500
MAX_TLS_STREAM_BYTES = 256 * 1024
TOKEN_SPLIT = re.compile(r"[.\-]")

RAW_FIELDS = {
    "dhcp": ("opt55", "opt60"),
    "dns": ("qname",),
    "mdns": ("qname",),
    "tls": ("version", "ciphers", "alpn", "sni"),
}
# Cột cờ có-mặt nằm ở `sdc_schema` vì đó là lược đồ chung, không riêng của bước parse.
# Gác này bắt trường hợp hai nơi kể tên nguồn lệch nhau — im lặng thì `n_sources` sai.
assert tuple(RAW_FIELDS) == tuple(SOURCES), (
    f"RAW_FIELDS {tuple(RAW_FIELDS)} lệch SOURCES {tuple(SOURCES)}"
)

DHCP_MSG_TYPES = {
    1: "DISCOVER", 2: "OFFER", 3: "REQUEST", 4: "DECLINE",
    5: "ACK", 6: "NAK", 7: "RELEASE", 8: "INFORM",
}
DNS_QTYPES = {1: "A", 28: "AAAA", 12: "PTR", 16: "TXT", 33: "SRV",
              5: "CNAME", 255: "ANY"}
DNS_PORTS = (53, 5353)


def _require_scapy():
    if PcapReader is None:
        raise RuntimeError("Packet extraction requires Scapy; install it with 'pip install scapy'")


def _is_missing(value):
    if value is None:
        return True
    try:
        missing = pd.isna(value)
        return bool(missing) if isinstance(missing, (bool, np.bool_)) else False
    except (TypeError, ValueError):
        return False


def _text(value, *, lowercase=False):
    if _is_missing(value):
        return None
    if isinstance(value, bytes):
        value = value.decode(errors="replace")
    value = str(value).strip()
    if not value:
        return None
    return value.lower() if lowercase else value


def normalize_mac(value):
    """Return a lowercase colon-separated six-byte MAC, or ``None``."""
    if isinstance(value, bytes):
        raw = value[:6].hex()
    else:
        raw = re.sub(r"[^0-9a-f]", "", str(value or "").lower())[:12]
    if len(raw) != 12 or raw == "0" * 12:
        return None
    return ":".join(raw[index:index + 2] for index in range(0, 12, 2))


def packet_source_mac(pkt):
    _require_scapy()
    if not pkt.haslayer(Ether):
        return None
    return normalize_mac(pkt[Ether].src)


def bootp_client_mac(pkt):
    _require_scapy()
    if not pkt.haslayer(BOOTP):
        return None
    return normalize_mac(bytes(pkt[BOOTP].chaddr or b"")[:6])


def packet_from_device(pkt, device_mac):
    """Whether the Ethernet source belongs to the requested device."""
    if device_mac is None:
        return True
    return packet_source_mac(pkt) == normalize_mac(device_mac)


def dhcp_matches_device(pkt, device_mac):
    """DHCP may match by Ethernet source or by BOOTP ``chaddr`` after a relay."""
    if device_mac is None:
        return True
    wanted = normalize_mac(device_mac)
    return packet_source_mac(pkt) == wanted or bootp_client_mac(pkt) == wanted


def _dhcp_options_dict(dhcp_layer):
    return {option[0]: option[1] for option in dhcp_layer.options
            if isinstance(option, tuple) and len(option) >= 2}


def extract_dhcp(pkt, ts=None):
    """Extract DHCP client identity fields in canonical and training-compatible names."""
    _require_scapy()
    if not pkt.haslayer(DHCP):
        return None
    options = _dhcp_options_dict(pkt[DHCP])
    hostname = _text(options.get("hostname"), lowercase=True)
    # Preserve VCI bytes for compatibility with exact L1 keys. The text encoder
    # performs its own lowercase normalization for L2.
    vendor_class_id = _text(options.get("vendor_class_id"))

    request_list = options.get("param_req_list")
    if isinstance(request_list, bytes):
        request_list = list(request_list)
    if request_list is not None and not isinstance(request_list, (list, tuple)):
        request_list = [request_list]
    param_req_list = (",".join(str(int(option)) for option in request_list)
                      if request_list else None)
    if param_req_list is None and vendor_class_id is None:
        return None

    msg_type_code = options.get("message-type")
    try:
        msg_type_code = int(msg_type_code)
    except (TypeError, ValueError):
        pass
    timestamp = float(pkt.time) if ts is None and hasattr(pkt, "time") else ts
    client_mac = bootp_client_mac(pkt)
    return {
        "proto": "dhcp",
        "opt55": param_req_list,
        "opt60": vendor_class_id,
        "ts": timestamp,
        "src_mac": packet_source_mac(pkt),
        "client_mac": client_mac,
        "msg_type_code": msg_type_code,
        "msg_type": DHCP_MSG_TYPES.get(msg_type_code, msg_type_code),
        "hostname": hostname,
        "vendor_class_id": vendor_class_id,
        "param_req_list": param_req_list,
    }


def _network_addresses(pkt):
    if IP is not None and pkt.haslayer(IP):
        return str(pkt[IP].src), str(pkt[IP].dst)
    if IPv6 is not None and pkt.haslayer(IPv6):
        return str(pkt[IPv6].src), str(pkt[IPv6].dst)
    return None, None

In [12]:
def extract_dns(pkt, ts=None):
    """Extract DNS and mDNS queries; responses are never feature evidence."""
    _require_scapy()
    if not pkt.haslayer(DNS):
        return []
    transport = pkt[UDP] if pkt.haslayer(UDP) else (pkt[TCP] if pkt.haslayer(TCP) else None)
    if transport is None:
        return []
    ports = (int(transport.sport), int(transport.dport))
    if not any(port in DNS_PORTS for port in ports):
        return []

    dns = pkt[DNS]
    if bool(dns.qr):
        return []
    is_mdns = 5353 in ports
    src_ip, dst_ip = _network_addresses(pkt)
    timestamp = float(pkt.time) if ts is None and hasattr(pkt, "time") else ts
    rows = []
    questions = list(dns.qd or []) if isinstance(dns.qd, (list, tuple)) else [dns.qd]
    for question in questions:
        if question is None:
            continue
        qname = _text(question.qname, lowercase=True)
        if not qname:
            continue
        qname = qname.rstrip(".")
        rows.append({
            "proto": "mdns" if is_mdns else "dns",
            "qname": qname,
            "ts": timestamp,
            "src_ip": src_ip,
            "dst_ip": dst_ip,
            "is_mdns": is_mdns,
            "is_response": False,
            "qry_name": qname,
            "qry_type": DNS_QTYPES.get(question.qtype, question.qtype),
            "rcode": None,
        })
    return rows


def is_grease(value):
    """RFC 8701 GREASE values have both bytes equal and a low nibble of 0xA."""
    try:
        value = int(value)
    except (TypeError, ValueError):
        return False
    return 0 <= value <= 0xFFFF and (value & 0x0F0F) == 0x0A0A


def normalize_cipher_suites(value):
    if _is_missing(value):
        return MISSING
    values = value if isinstance(value, (list, tuple)) else re.findall(r"\d+", str(value))
    normalized = []
    for item in values:
        try:
            number = int(item)
        except (TypeError, ValueError):
            continue
        if not is_grease(number):
            normalized.append(str(number))
    return ",".join(normalized) if normalized else MISSING


def normalize_tls_version(value):
    if _is_missing(value):
        return MISSING
    text = str(value).strip().lower()
    if text.startswith("0x"):
        text = text[2:]
    if re.fullmatch(r"\d{1,4}", text):
        return text.zfill(4)
    return MISSING


def parse_tls_client_hello(raw):
    """Parse a complete TLS ClientHello record and remove GREASE from its fingerprint."""
    try:
        raw = bytes(raw)
        if len(raw) < 9 or raw[0] != 0x16:
            return None
        record_length = struct.unpack("!H", raw[3:5])[0]
        if record_length > MAX_TLS_STREAM_BYTES or len(raw) < 5 + record_length:
            return None
        handshake = raw[5:5 + record_length]
        if len(handshake) < 4 or handshake[0] != 0x01:
            return None
        handshake_length = int.from_bytes(handshake[1:4], "big")
        if len(handshake) < 4 + handshake_length:
            return None
        body = handshake[4:4 + handshake_length]

        position = 0
        if len(body) < 35:
            return None
        version = body[position:position + 2].hex()
        position += 34  # version + random
        session_length = body[position]
        position += 1
        if position + session_length + 2 > len(body):
            return None
        position += session_length

        cipher_length = struct.unpack("!H", body[position:position + 2])[0]
        position += 2
        if cipher_length % 2 or position + cipher_length > len(body):
            return None
        cipher_suites = [struct.unpack("!H", body[offset:offset + 2])[0]
                         for offset in range(position, position + cipher_length, 2)]
        cipher_suites = [value for value in cipher_suites if not is_grease(value)]
        position += cipher_length

        if position >= len(body):
            return None
        compression_length = body[position]
        position += 1
        if position + compression_length > len(body):
            return None
        position += compression_length

        sni, alpn = None, []
        if position + 2 <= len(body):
            extensions_length = struct.unpack("!H", body[position:position + 2])[0]
            position += 2
            extensions_end = position + extensions_length
            if extensions_end > len(body):
                return None
            while position + 4 <= extensions_end:
                extension_type, extension_length = struct.unpack(
                    "!HH", body[position:position + 4]
                )
                position += 4
                if position + extension_length > extensions_end:
                    return None
                data = body[position:position + extension_length]
                position += extension_length

                if extension_type == 0 and len(data) >= 5:
                    names_length = struct.unpack("!H", data[:2])[0]
                    cursor = 2
                    names_end = min(len(data), 2 + names_length)
                    while cursor + 3 <= names_end:
                        name_type = data[cursor]
                        name_length = struct.unpack("!H", data[cursor + 1:cursor + 3])[0]
                        cursor += 3
                        if cursor + name_length > names_end:
                            break
                        if name_type == 0:
                            sni = _text(data[cursor:cursor + name_length], lowercase=True)
                            break
                        cursor += name_length
                elif extension_type == 16 and len(data) >= 2:
                    protocols_length = struct.unpack("!H", data[:2])[0]
                    cursor = 2
                    protocols_end = min(len(data), 2 + protocols_length)
                    while cursor < protocols_end:
                        protocol_length = data[cursor]
                        cursor += 1
                        if cursor + protocol_length > protocols_end:
                            break
                        protocol = _text(data[cursor:cursor + protocol_length], lowercase=True)
                        if protocol:
                            alpn.append(protocol)
                        cursor += protocol_length

        ciphers = ",".join(str(value) for value in cipher_suites)
        if not ciphers:
            return None
        alpn_value = ",".join(alpn) if alpn else None
        return {
            "proto": "tls",
            "version": version,
            "ciphers": ciphers,
            "alpn": alpn_value,
            "sni": sni,
            "tls_version": version,
            "cipher_suites": ciphers,
            "n_cipher_suites": len(cipher_suites),
        }
    except (IndexError, TypeError, ValueError, struct.error):
        return None


def _tls_packet_metadata(pkt, ts):
    src_ip, dst_ip = _network_addresses(pkt)
    return {
        "ts": float(pkt.time) if ts is None and hasattr(pkt, "time") else ts,
        "src_ip": src_ip,
        "dst_ip": dst_ip,
        "dst_port": int(pkt[TCP].dport),
    }

In [13]:
class TLSStreamReassembler:
    """Small bounded TCP reassembler for outbound TLS ClientHello records."""

    def __init__(self, max_stream_bytes=MAX_TLS_STREAM_BYTES):
        self.max_stream_bytes = max_stream_bytes
        self._segments = {}
        self._emitted = set()

    @staticmethod
    def _flow_key(pkt):
        src_ip, dst_ip = _network_addresses(pkt)
        if src_ip is None:
            src_ip, dst_ip = packet_source_mac(pkt), None
        tcp = pkt[TCP]
        return src_ip, dst_ip, int(tcp.sport), int(tcp.dport)

    @staticmethod
    def _assemble(segments):
        chunks = []
        cursor = None
        for sequence, payload in sorted(segments.items()):
            if cursor is None:
                cursor = sequence
            if sequence > cursor:
                break
            offset = max(0, cursor - sequence)
            if offset < len(payload):
                chunks.append(payload[offset:])
                cursor += len(payload) - offset
        return b"".join(chunks)

    def feed(self, pkt, ts=None):
        _require_scapy()
        if not (pkt.haslayer(TCP) and pkt.haslayer(Raw)):
            return []
        payload = bytes(pkt[Raw].load)
        if not payload:
            return []
        flow = self._flow_key(pkt)
        segments = self._segments.setdefault(flow, {})
        sequence = int(pkt[TCP].seq)
        if len(payload) > len(segments.get(sequence, b"")):
            segments[sequence] = payload
        if sum(map(len, segments.values())) > self.max_stream_bytes:
            self._segments.pop(flow, None)
            return []

        stream = self._assemble(segments)
        found = []
        offset = 0
        while offset + 5 <= len(stream):
            if stream[offset] != 0x16 or stream[offset + 1] != 0x03:
                offset += 1
                continue
            record_length = struct.unpack("!H", stream[offset + 3:offset + 5])[0]
            total = 5 + record_length
            if record_length > self.max_stream_bytes:
                offset += 1
                continue
            if offset + total > len(stream):
                break
            record = stream[offset:offset + total]
            parsed = parse_tls_client_hello(record)
            if parsed is not None:
                digest = hashlib.sha256(record).digest()
                marker = (flow, digest)
                if marker not in self._emitted:
                    self._emitted.add(marker)
                    parsed.update(_tls_packet_metadata(pkt, ts))
                    found.append(parsed)
            offset += total
        return found


def extract_tls(pkt, ts=None, reassembler=None):
    """Extract one ClientHello; pass a reassembler to support TCP segmentation."""
    _require_scapy()
    if not (pkt.haslayer(TCP) and pkt.haslayer(Raw)):
        return None
    if reassembler is not None:
        rows = reassembler.feed(pkt, ts)
        return rows[0] if rows else None
    parsed = parse_tls_client_hello(bytes(pkt[Raw].load))
    if parsed is not None:
        parsed.update(_tls_packet_metadata(pkt, ts))
    return parsed


def infer_device_mac(pcap_path):
    """Infer the most frequent DHCP client MAC for a single-device capture."""
    _require_scapy()
    candidates = Counter()
    with PcapReader(str(pcap_path)) as reader:
        for pkt in reader:
            row = extract_dhcp(pkt)
            if row and row["client_mac"]:
                candidates[row["client_mac"]] += 1
    return candidates.most_common(1)[0][0] if candidates else None


def extract_records(packets, device_mac=None, max_dns=MAX_DNS_PER_WINDOW):
    """Extract canonical records from packets belonging to one device/window."""
    _require_scapy()
    device_mac = normalize_mac(device_mac) if device_mac is not None else None
    records, counts = [], Counter()
    seen_names = set()
    reassembler = TLSStreamReassembler()

    for pkt in packets:
        if not pkt.haslayer(Ether):
            continue
        timestamp = float(pkt.time) if hasattr(pkt, "time") else None
        dhcp = extract_dhcp(pkt, timestamp)
        if dhcp is not None and dhcp_matches_device(pkt, device_mac):
            records.append(dhcp)
            counts["dhcp"] += 1

        if not packet_from_device(pkt, device_mac):
            continue
        for row in extract_dns(pkt, timestamp):
            key = row["proto"], row["qname"]
            if key in seen_names or counts["dns"] + counts["mdns"] >= max_dns:
                continue
            seen_names.add(key)
            records.append(row)
            counts[row["proto"]] += 1

        for row in reassembler.feed(pkt, timestamp):
            records.append(row)
            counts["tls"] += 1
    return records, counts


def read_pcap(pcap_path, device_mac=None, max_dns=MAX_DNS_PER_WINDOW):
    """Read one pcap as a device window, optionally inferring its DHCP client MAC."""
    _require_scapy()
    pcap_path = Path(pcap_path)
    if device_mac == "auto":
        device_mac = infer_device_mac(pcap_path)
        if device_mac is None:
            raise ValueError(f"Cannot infer device MAC from {pcap_path}")
    with PcapReader(str(pcap_path)) as reader:
        return extract_records(reader, device_mac=device_mac, max_dns=max_dns)


def domain_tokens(names, max_tokens=MAX_TOKENS):
    """Domain names to unique lowercase tokens, preserving first-seen order."""
    tokens = []
    for name in names:
        if _is_missing(name):
            continue
        normalized = str(name).lower().rstrip(".")
        tokens.extend(token for token in TOKEN_SPLIT.split(normalized) if token)
    unique = list(dict.fromkeys(tokens))[:max_tokens]
    return " ".join(unique) if unique else MISSING


def _mode(values):
    clean = [str(value) for value in values
             if not _is_missing(value) and str(value).strip() not in ("", MISSING)]
    if not clean:
        return MISSING
    return pd.Series(clean).mode().iat[0]


def _tls_fingerprint(record):
    version = normalize_tls_version(record.get("version", record.get("tls_version")))
    ciphers = normalize_cipher_suites(record.get("ciphers", record.get("cipher_suites")))
    alpn = _text(record.get("alpn"), lowercase=True) or ""
    if MISSING in (version, ciphers):
        return MISSING
    return f"{version}|{ciphers}|{alpn}"

In [14]:
def aggregate(records, feature_cols):
    """Aggregate canonical records for one device/window into one model feature row."""
    by_source = {source: [] for source in RAW_FIELDS}
    for record in records:
        if not isinstance(record, dict):
            raise TypeError("Each raw record must be a dictionary")
        proto = record.get("proto")
        if proto not in by_source:
            raise ValueError(f"Invalid proto {proto!r}; expected one of {sorted(RAW_FIELDS)}")
        by_source[proto].append(record)

    row = {}
    row["dhcp_prl"] = _mode(record.get("opt55", record.get("param_req_list"))
                             for record in by_source["dhcp"])
    row["dhcp_vci"] = _mode(record.get("opt60", record.get("vendor_class_id"))
                             for record in by_source["dhcp"])
    options = set()
    if row["dhcp_prl"] != MISSING:
        options = {int(value) for value in re.findall(r"\d+", row["dhcp_prl"])}
    for column in feature_cols:
        if column.startswith("dhcp_opt_"):
            row[column] = int(int(column.rsplit("_", 1)[1]) in options)
    row["dhcp_prl_len"] = len(options)

    row["dns_tokens"] = domain_tokens(record.get("qname", record.get("qry_name"))
                                         for record in by_source["dns"])
    row["mdns_tokens"] = domain_tokens(record.get("qname", record.get("qry_name"))
                                          for record in by_source["mdns"])

    tls_records = by_source["tls"]
    row["tls_fp"] = _mode(_tls_fingerprint(record) for record in tls_records)
    row["tls_version"] = _mode(
        normalize_tls_version(record.get("version", record.get("tls_version")))
        for record in tls_records
    )
    row["tls_alpn"] = _mode(_text(record.get("alpn"), lowercase=True)
                              for record in tls_records)
    row["tls_ciphers"] = _mode(
        normalize_cipher_suites(record.get("ciphers", record.get("cipher_suites")))
        for record in tls_records
    )
    row["tls_sni_tokens"] = domain_tokens(record.get("sni") for record in tls_records)

    for source in RAW_FIELDS:
        row[f"has_{source}"] = int(bool(by_source[source]))
    row["n_sources"] = sum(row[flag] for flag in SOURCE_FLAGS)

    numeric_prefixes = ("has_", "dhcp_opt_")
    for column in feature_cols:
        if column not in row:
            row[column] = (0 if column.startswith(numeric_prefixes)
                           or column == "dhcp_prl_len" else MISSING)
    return row

### Dựng dataset đã xác minh

Validate production taxonomy and build sessions_verified.parquet.

The draft dataset is never overwritten.  This script remaps labels by canonical device,
drops only explicitly excluded devices, and fails closed on incomplete taxonomy.

In [15]:
import argparse
from pathlib import Path

import pandas as pd



# Tên có tiền tố `DEFAULT_` vì sau `%run` mọi module nằm chung một namespace: `SOURCE`
# hay `OUTPUT` trần là tên quá dễ đụng với module khác.
DEFAULT_SESSIONS = DATA / "sessions.parquet"
DEFAULT_LABELS = DATA / "device_labels_verified.csv"
DEFAULT_EXCLUDED = DATA / "device_labels_excluded.csv"
DEFAULT_OUTPUT = SESSIONS_PATH


def _read_csv(path: Path, required: set[str]) -> pd.DataFrame:
    frame = pd.read_csv(path, keep_default_na=False)
    frame = frame.rename(columns={old: new for old, new in HEAD_ALIASES.items()
                                  if old in frame.columns and new not in frame.columns})
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")
    return frame


def validate_taxonomy(labels: pd.DataFrame, excluded: pd.DataFrame) -> None:
    required = ["canonical_device", *LABEL_COLS]
    if labels[required].apply(lambda col: col.astype(str).str.strip().eq("")).any().any():
        raise ValueError("verified labels contain blank required values")
    if labels["canonical_device"].duplicated().any():
        duplicates = sorted(labels.loc[labels["canonical_device"].duplicated(False),
                                           "canonical_device"].unique())
        raise ValueError(f"duplicate verified devices: {duplicates}")
    if excluded["canonical_device"].duplicated().any():
        raise ValueError("excluded device list contains duplicates")
    overlap = set(labels["canonical_device"]) & set(excluded["canonical_device"])
    if overlap:
        raise ValueError(f"devices cannot be both verified and excluded: {sorted(overlap)}")
    unknown = labels["make"].astype(str).str.casefold().eq("unknown")
    if unknown.any():
        raise ValueError("make=Unknown is forbidden in the production taxonomy")

    model_make = labels.groupby("model")["make"].nunique()
    conflicts = model_make[model_make.ne(1)]
    if not conflicts.empty:
        raise ValueError(f"model maps to multiple makes: {conflicts.index.tolist()}")
    model_type = labels.groupby("model")["type"].nunique()
    conflicts = model_type[model_type.ne(1)]
    if not conflicts.empty:
        raise ValueError(f"model maps to multiple types: {conflicts.index.tolist()}")


def build_verified_dataset(
    source: Path = DEFAULT_SESSIONS,
    labels_path: Path = DEFAULT_LABELS,
    excluded_path: Path = DEFAULT_EXCLUDED,
    output: Path = DEFAULT_OUTPUT,
) -> pd.DataFrame:
    labels = _read_csv(labels_path, {"canonical_device", *LABEL_COLS})
    excluded = _read_csv(excluded_path, {"canonical_device", "reason"})
    validate_taxonomy(labels, excluded)

    sessions = pd.read_parquet(source)
    known = set(labels["canonical_device"])
    dropped = set(excluded["canonical_device"])
    present = set(sessions["canonical_device"].astype(str))
    unclassified = present - known - dropped
    if unclassified:
        raise ValueError(f"sessions contain unclassified devices: {sorted(unclassified)}")
    unused = known - present
    if unused:
        raise ValueError(f"verified labels contain devices absent from sessions: {sorted(unused)}")

    result = sessions[~sessions["canonical_device"].isin(dropped)].copy()
    # Nhãn luôn được gán lại từ taxonomy ở ngay dưới, nên cột nhãn mang tên cũ trong file
    # nguồn chỉ là tàn dư. Giữ lại thì dataset có cả `family` lẫn `model` cùng lúc.
    result = result.drop(columns=[c for c in HEAD_ALIASES if c in result.columns])
    mapping = labels.set_index("canonical_device")[LABEL_COLS]
    for column in LABEL_COLS:
        result[column] = result["canonical_device"].map(mapping[column])
    if result[LABEL_COLS].isna().any().any():
        raise ValueError("label mapping unexpectedly produced missing values")

    output.parent.mkdir(parents=True, exist_ok=True)
    result.to_parquet(output)
    print(
        f"wrote {output}: {len(result)} sessions, "
        f"{result.canonical_device.nunique()} devices; excluded={sorted(dropped)}"
    )
    return result


def prepare_parse_args(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--source", type=Path, default=DEFAULT_SESSIONS)
    parser.add_argument("--labels", type=Path, default=DEFAULT_LABELS)
    parser.add_argument("--excluded", type=Path, default=DEFAULT_EXCLUDED)
    parser.add_argument("--output", type=Path, default=DEFAULT_OUTPUT)
    return parser.parse_args(argv)


def prepare_main(argv=None):
    args = prepare_parse_args(argv)
    build_verified_dataset(args.source, args.labels, args.excluded, args.output)

## 3. Suy luận: hợp đồng model, Predictor, DeviceTracker


### Hợp đồng artifact — chọn run, kiểm bundle

Hop dong cua artifact: chon run nao, va bundle co hop le khong.

Tach khoi `sdc_predict` vi day la tang GAC - no chay xong truoc khi co bat ky du doan
nao, va no phai fail-closed. Thieu/hong policy thi model khong duoc nap, khong bao gio
lui ve nguong mac dinh 0.0.

In [16]:
import json
import math
from collections.abc import Mapping
from numbers import Real
from pathlib import Path


LEGACY_FORMAT = "sdc-legacy-v1"
TIERED_FORMAT = "sdc-tiered-v2"
MANIFEST_FORMAT = "sdc-current-model-v1"
CURRENT_MODEL_MANIFEST = MODELS / "current_model.json"


class ModelContractError(ValueError):
    """The artifact exists, but its runtime contract is missing or invalid."""


def read_json(path, purpose):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except FileNotFoundError as exc:
        raise ModelContractError(f"Missing {purpose}: {path}") from exc
    except (OSError, UnicodeError, json.JSONDecodeError) as exc:
        raise ModelContractError(f"Cannot read {purpose} {path}: {exc}") from exc


def resolve_run_dir(run_dir=None, manifest_path=CURRENT_MODEL_MANIFEST):
    """Resolve an explicit run or the pinned run in ``current_model.json``.

    Directory-name ordering is deliberately not a model-selection mechanism.
    """
    if run_dir is not None:
        selected = Path(run_dir)
        # A bare run id is resolved under Models; explicit relative/absolute paths
        # retain their normal path semantics.
        if (not selected.is_absolute() and len(selected.parts) == 1
                and not selected.is_dir()):
            selected = MODELS / selected
    else:
        manifest_path = Path(manifest_path)
        doc = read_json(manifest_path, "current-model manifest")
        if not isinstance(doc, Mapping) or doc.get("format") != MANIFEST_FORMAT:
            raise ModelContractError(
                f"Invalid manifest format in {manifest_path}; expected {MANIFEST_FORMAT!r}"
            )
        run = doc.get("run")
        if not isinstance(run, str) or not run.strip():
            raise ModelContractError(f"Manifest {manifest_path} must contain a non-empty 'run'")
        relative = Path(run)
        if relative.is_absolute():
            raise ModelContractError(f"Manifest run must be relative to {manifest_path.parent}")
        base = manifest_path.parent.resolve()
        selected = (base / relative).resolve()
        try:
            selected.relative_to(base)
        except ValueError as exc:
            raise ModelContractError("Manifest run must not escape the Models directory") from exc

    selected = selected.resolve()
    if not selected.is_dir():
        raise FileNotFoundError(f"Model run directory does not exist: {selected}")
    if not (selected / "model.joblib").is_file():
        raise FileNotFoundError(f"Missing model artifact: {selected / 'model.joblib'}")
    return selected


def _probability(value, name):
    if isinstance(value, bool) or not isinstance(value, Real):
        raise ModelContractError(f"{name} must be a number in [0, 1]")
    value = float(value)
    if not math.isfinite(value) or not 0.0 <= value <= 1.0:
        raise ModelContractError(f"{name} must be a finite number in [0, 1]")
    return value


def validate_thresholds(raw, heads, name):
    if not isinstance(raw, Mapping):
        raise ModelContractError(f"{name} must be an object")
    missing = [head for head in heads if head not in raw]
    if missing:
        raise ModelContractError(f"{name} is missing heads: {missing}")
    return {head: _probability(raw[head], f"{name}.{head}") for head in heads}


def validate_tiered_policy(bundle, heads):
    raw_thresholds = bundle.get("thresholds_by_source")
    raw_min_sources = bundle.get("min_sources")
    if not isinstance(raw_thresholds, Mapping):
        raise ModelContractError("thresholds_by_source must be an object")
    if not isinstance(raw_min_sources, Mapping):
        raise ModelContractError("min_sources must be an object")

    thresholds = {}
    min_sources = {}
    max_sources = len(SOURCE_FLAGS)
    for head in heads:
        minimum = raw_min_sources.get(head)
        if isinstance(minimum, bool) or not isinstance(minimum, int):
            raise ModelContractError(f"min_sources.{head} must be an integer")
        if not 1 <= minimum <= max_sources:
            raise ModelContractError(
                f"min_sources.{head} must be between 1 and {max_sources}"
            )
        min_sources[head] = minimum
        for n_sources in range(1, max_sources + 1):
            key = f"{head}|{n_sources}"
            if key not in raw_thresholds:
                raise ModelContractError(f"thresholds_by_source is missing {key!r}")
            thresholds[key] = _probability(
                raw_thresholds[key], f"thresholds_by_source.{key}"
            )

    l1_floor = _probability(bundle.get("l1_floor"), "l1_floor")
    return thresholds, min_sources, l1_floor


def validate_source_combo_thresholds(bundle, heads):
    raw = bundle.get("thresholds_by_source_combo", {})
    if not isinstance(raw, Mapping):
        raise ModelContractError("thresholds_by_source_combo must be an object")
    valid_sources = [flag.removeprefix("has_") for flag in SOURCE_FLAGS]
    thresholds = {}
    for key, value in raw.items():
        if not isinstance(key, str) or "|" not in key:
            raise ModelContractError(f"invalid source-combo threshold key: {key!r}")
        head, combo = key.split("|", 1)
        sources = combo.split("+") if combo else []
        canonical = "+".join(source for source in valid_sources if source in sources)
        if head not in heads or not sources or len(sources) != len(set(sources)):
            raise ModelContractError(f"invalid source-combo threshold key: {key!r}")
        if any(source not in valid_sources for source in sources) or combo != canonical:
            raise ModelContractError(f"invalid source combination: {combo!r}")
        thresholds[key] = _probability(value, f"thresholds_by_source_combo.{key}")
    return thresholds

### `Predictor` — quyết định cho một cửa sổ

`Predictor` - quyet dinh cho MOT cua so thu thap.

    predictor = Predictor(ROOT / "Models" / "<run_id>")
    predictor.predict(records)

Thu tu cot, vocab va label map deu doc tu `model.joblib` chu khong suy lai o day - suy
lai la cach chac chan nhat de encoder luc train va luc chay lech nhau.

**Ba trang thai, khong phai hai.** Mot cua so khong tra loi duoc vi hai ly do khac han
nhau, va gop chung lai la sai:

| Trang thai | Nghia | Hanh dong |
|---|---|---|
| `answer` | co nhan kem confidence | dung |
| `unknown` | chua tung thay van tay nay, hoac model khong du tu tin | **ung vien thu nap** |
| `ambiguous` | biet van tay, nhung no ung voi nhieu nhan | **dung thu nap** |

Gom nhieu cua so cua cung mot MAC: xem `sdc_track.DeviceTracker`.

In [17]:
import json
import logging
from collections.abc import Mapping

import joblib
import numpy as np
import pandas as pd


DECISION_LOGGER = logging.getLogger(f"{__name__}.decision")

In [18]:
class Predictor:
    """Nạp một run đã train rồi trả lời cho từng cửa sổ thu thập."""

    # Điểm tối thiểu để một luật catalog được phép LẬT một câu trả lời L2 sát ngưỡng.
    # 4.0 là trọng số của signature đặc trưng dòng sản phẩm trong
    # `Data/device_type_rules.json` (`dsp…` -> Smart Plug, `dchg…` -> Smart Hub) — thứ
    # khớp thẳng vào chuỗi chính thiết bị tự khai. Luật yếu hơn (2.0–3.0) vẫn chỉ được
    # điền vào chỗ model đã abstain, y như trước.
    SEMANTIC_OVERRIDE_SCORE = 4.0

    def __init__(self, run_dir=None, thresholds=None, enrolled=None, hierarchy=None,
                 manifest_path=CURRENT_MODEL_MANIFEST,
                 semantic_catalog=DEFAULT_CATALOG):
        run_dir = resolve_run_dir(run_dir, manifest_path)
        try:
            bundle = joblib.load(run_dir / "model.joblib")
        except Exception as exc:
            raise ModelContractError(f"Cannot load model artifact in {run_dir}: {exc}") from exc
        if not isinstance(bundle, Mapping):
            raise ModelContractError("model.joblib must contain a dictionary bundle")

        required = ("models", "encoder", "l1_tables")
        missing = [key for key in required if key not in bundle]
        if missing:
            raise ModelContractError(f"Model bundle is missing required fields: {missing}")
        if not isinstance(bundle["models"], Mapping) or not bundle["models"]:
            raise ModelContractError("models must be a non-empty object")

        raw_format = bundle.get("format") if "format" in bundle else None
        if raw_format is None:
            contract_format = LEGACY_FORMAT
        elif raw_format in (LEGACY_FORMAT, TIERED_FORMAT):
            contract_format = raw_format
        else:
            raise ModelContractError(f"Unsupported model format: {raw_format!r}")

        self.run_dir = run_dir
        self.contract_format = contract_format
        self.models = bundle["models"]
        self.encoder = bundle["encoder"]
        self.heads = list(self.models)
        enc = self.encoder
        try:
            self.feature_cols = enc["num_cols"] + enc["cat_cols"] + enc["text_cols"]
        except (KeyError, TypeError) as exc:
            raise ModelContractError("encoder is missing num_cols/cat_cols/text_cols") from exc

        # Dựng lại bảng L1 từ bundle. `ambiguous` có thể vắng ở run cũ -> coi như rỗng.
        self.table = FingerprintTable()
        self.table.tables = bundle["l1_tables"]
        self.table.ambiguous = bundle.get("l1_ambiguous", {})
        self.table.load_enrolled(read_enrolled() if enrolled is None else enrolled)

        # Ngưỡng abstain do tầng train (lib_4) hiệu chỉnh từ OOF, nằm trong chính bundle.
        if contract_format == TIERED_FORMAT:
            if thresholds is not None:
                raise ModelContractError(
                    "sdc-tiered-v2 policy must come from the bundle; thresholds override is not allowed"
                )
            (self.thresholds_by_source,
             self.min_sources,
             self.l1_floor) = validate_tiered_policy(bundle, self.heads)
            self.thresholds_by_source_combo = validate_source_combo_thresholds(
                bundle, self.heads
            )
            self.thresholds = dict(self.thresholds_by_source)
        else:
            if thresholds is None:
                doc = read_json(run_dir / "thresholds.json", "legacy threshold policy")
                if not isinstance(doc, Mapping) or "thresholds" not in doc:
                    raise ModelContractError(
                        f"Legacy policy {run_dir / 'thresholds.json'} must contain 'thresholds'"
                    )
                thresholds = doc["thresholds"]
            self.thresholds = validate_thresholds(
                thresholds, self.heads, "legacy thresholds"
            )
            self.thresholds_by_source = None
            self.thresholds_by_source_combo = {}
            self.min_sources = {head: 0 for head in self.heads}
            self.l1_floor = None

        # model -> (make, tập type cho phép). `Tuya Plug` ứng với 3 type nên vế type
        # phải là tập cho phép chứ không phải đẳng thức — xem mục 4 notebook 04.
        # Head model có thể mang tên cũ `family` nếu bundle được train trước lần đổi tên.
        # Đọc từ chính bundle chứ không giả định, để model cũ vẫn nạp được nguyên vẹn.
        self.model_head = next(
            (head for head in (MODEL_HEAD, LEGACY_MODEL_HEAD) if head in self.heads),
            None,
        )
        self.hierarchy = hierarchy if hierarchy is not None else bundle.get("hierarchy", {})
        if not isinstance(self.hierarchy, Mapping):
            raise ModelContractError("hierarchy must be an object")
        self.semantic_type = (
            SemanticTypeClassifier(semantic_catalog)
            if semantic_catalog is not None else None
        )

    # --- Một cửa sổ ------------------------------------------------------------------

    def _model_scores(self, head, X, top_k):
        if isinstance(top_k, bool) or not isinstance(top_k, int) or top_k < 1:
            raise ValueError("top_k must be a positive integer")
        model = self.models[head]
        proba = np.asarray(model.predict_proba(X), dtype=float)
        classes = np.asarray(model.classes_)
        if proba.ndim != 2 or proba.shape[0] != 1 or proba.shape[1] != len(classes):
            raise ModelContractError(f"Invalid predict_proba output for head {head!r}")
        if len(classes) == 0 or not np.isfinite(proba).all():
            raise ModelContractError(f"Invalid class probabilities for head {head!r}")

        order = np.argsort(proba[0])[::-1]
        best_index = int(order[0])
        second_index = int(order[1]) if len(order) > 1 else None
        best = float(proba[0, best_index])
        second = float(proba[0, second_index]) if second_index is not None else 0.0
        return {
            "proba": proba[0],
            "classes": classes,
            "best": best,
            "model_top1": str(classes[best_index]),
            "model_top2": str(classes[second_index]) if second_index is not None else None,
            "margin": best - second,
            "topk": [(str(classes[i]), round(float(proba[0, i]), 4))
                     for i in order[:top_k]],
        }

    @staticmethod
    def _l1_backing(scores, label):
        labels = [str(value) for value in scores["classes"]]
        try:
            index = labels.index(str(label))
        except ValueError:
            return None
        return float(scores["proba"][index])

    def _threshold_for(self, head, n_sources, source_combo=None):
        if self.contract_format == TIERED_FORMAT:
            combo_key = f"{head}|{source_combo}"
            if source_combo and combo_key in self.thresholds_by_source_combo:
                return self.thresholds_by_source_combo[combo_key]
            return self.thresholds_by_source[f"{head}|{n_sources}"]
        return self.thresholds[head]

    def _has_policy_violation(self, head, result):
        if result["status"] != "answer":
            return False
        if result["source"] == "model":
            threshold = result.get("threshold_used")
            return (threshold is None
                    or result["confidence"] + 0.00005 < threshold
                    or result["n_sources"] < self.min_sources[head])
        if result["source"] == "semantic":
            threshold = result.get("threshold_used")
            return (threshold is None
                    or result.get("confidence", 0.0) + 0.00005 < threshold)
        if result["source"] == "mined" and self.contract_format == TIERED_FORMAT:
            return (result.get("l1_backing") is None
                    or result["l1_backing"] + 0.00005 < self.l1_floor)
        return False

    @staticmethod
    def _audit_decision(result):
        fields = (
            "head", "n_sources", "confidence", "threshold_used", "top1", "top2",
            "margin", "l1_status", "hierarchy_status", "decision_reason",
            "policy_violation", "source_combo", "open_vocabulary", "semantic_score",
            "catalog_version", "semantic_override", "overridden_top1",
        )
        DECISION_LOGGER.info(
            "%s", json.dumps({key: result.get(key) for key in fields},
                              ensure_ascii=False, sort_keys=True)
        )

    def predict_row(self, row, top_k=3, records=None):
        """Apply L1/L2 policy and return an auditable decision for every head."""
        frame = pd.DataFrame([row])[self.feature_cols]
        X, _, _ = apply_encoder(frame, self.encoder)
        n_sources = sum(int(row.get(flag) == 1) for flag in SOURCE_FLAGS)
        source_combo = "+".join(
            flag.removeprefix("has_") for flag in SOURCE_FLAGS if row.get(flag) == 1
        ) or "none"

        out = {"input_dim": int(X.shape[1]), "n_sources": n_sources,
               "contract_format": self.contract_format}
        for head in self.heads:
            found = self.table.lookup(row, head)
            scores = self._model_scores(head, X, top_k)
            l1_status = (f"hit_{found['source']}" if found["status"] == "hit"
                         else found["status"])
            result = {
                "head": head,
                "status": "abstain",
                "fp": found["status"],
                "l1_status": l1_status,
                "candidates": found["labels"] or None,
                "top1": None,
                "top2": scores["model_top2"],
                "candidate_top1": scores["model_top1"],
                "confidence": round(scores["best"], 4),
                "threshold_used": None,
                "margin": round(scores["margin"], 4),
                "l1_backing": None,
                "n_sources": n_sources,
                "source_combo": source_combo,
                "min_sources": self.min_sources[head],
                "retrieval_mode": "model",
                "source": "model",
                "enrollable": found["status"] == "miss",
                "topk": scores["topk"],
                "hierarchy_status": "pending",
                "decision_reason": None,
                "conflict": False,
            }

            if found["status"] == "hit" and found["source"] == "enrolled":
                result.update(
                    status="answer",
                    top1=str(found["label"]),
                    confidence=1.0,
                    retrieval_mode=found["mode"],
                    source="enrolled",
                    enrollable=False,
                    topk=[(str(found["label"]), 1.0)],
                    decision_reason="l1_enrolled_hit",
                )
            elif found["status"] == "hit":
                backing = self._l1_backing(scores, found["label"])
                result.update(
                    confidence=round(backing, 4) if backing is not None else None,
                    l1_backing=round(backing, 4) if backing is not None else None,
                    retrieval_mode=found["mode"],
                    source="mined",
                    enrollable=False,
                )
                if self.contract_format == LEGACY_FORMAT:
                    result.update(
                        status="answer",
                        top1=str(found["label"]),
                        confidence=1.0,
                        topk=[(str(found["label"]), 1.0)],
                        decision_reason="l1_legacy_hit",
                    )
                else:
                    result["threshold_used"] = self.l1_floor
                    if backing is None:
                        result["decision_reason"] = "l1_label_not_in_model"
                    elif backing < self.l1_floor:
                        result["decision_reason"] = "l1_backing_below_floor"
                    else:
                        result.update(
                            status="answer",
                            top1=str(found["label"]),
                            topk=[(str(found["label"]), round(backing, 4))],
                            decision_reason="l1_backing_met",
                        )
            else:
                enough_sources = (self.contract_format == LEGACY_FORMAT
                                  or n_sources >= self.min_sources[head])
                if not enough_sources:
                    result["decision_reason"] = "insufficient_sources"
                else:
                    threshold = self._threshold_for(head, n_sources, source_combo)
                    result["threshold_used"] = threshold
                    if scores["best"] < threshold:
                        result["decision_reason"] = "confidence_below_threshold"
                    else:
                        result.update(
                            status="answer",
                            top1=scores["model_top1"],
                            decision_reason="l2_threshold_met",
                        )

            out[head] = result

        self._apply_hierarchy(out)
        semantic = (self.semantic_type.classify(row=row, records=records)
                    if self.semantic_type is not None else None)
        out["semantic_type"] = semantic
        self._apply_semantic_type(out, semantic)
        for head in self.heads:
            out[head]["policy_violation"] = self._has_policy_violation(head, out[head])
            self._audit_decision(out[head])
        return out

    def predict(self, records, top_k=3):
        """records: sự kiện raw của 1 MAC trong 1 cửa sổ. Trả dict theo output contract."""
        return self.predict_row(
            aggregate(records, self.feature_cols), top_k=top_k, records=records
        )

    def _apply_semantic_type(self, out, semantic):
        """Điền `type` từ catalog khi model abstain, hoặc lật một câu trả lời L2 yếu.

        Trên thiết bị chưa từng thấy, L2 chỉ chọn được trong các lớp đã học và
        confidence sát ngưỡng của nó là nhiễu, trong khi một signature đặc trưng dòng
        sản phẩm là bằng chứng trực tiếp. Đo trên IoT Sentinel: `dsp3d6f.local` có luật
        -> Smart Plug và `dchgfd4e.local` có luật -> Smart Hub, nhưng cả hai bị một câu
        trả lời L2 `Smart Lamp` ở confidence 0.52 / ngưỡng 0.48 chặn lại.

        Chỉ lật câu trả lời đến từ **ngưỡng L2**. Vân tay L1 (mined/enrolled) vẫn thắng
        vì đó là khớp chính xác; head `model` đã trả lời cũng vẫn thắng, vì hạ `type`
        lúc đó sẽ phá đúng tính nhất quán mà `_apply_hierarchy` vừa kiểm xong.
        """
        if not semantic or semantic.get("status") != "answer":
            return
        result = out.get("type")
        model_result = out.get(self.model_head, {}) if self.model_head else {}
        if not result:
            return
        if (result.get("hierarchy_status") == "conflict"
                or model_result.get("status") == "answer"):
            return
        overrode = result.get("status") == "answer"
        if overrode:
            if result.get("decision_reason") != "l2_threshold_met":
                return
            if float(semantic.get("score") or 0.0) < self.SEMANTIC_OVERRIDE_SCORE:
                return
        candidates = semantic.get("candidates", [])
        previous_top1 = result.get("top1")
        result.update(
            semantic_override=overrode,
            overridden_top1=previous_top1 if overrode else None,
            status="answer",
            top1=semantic["type"],
            top2=(candidates[1]["type"] if len(candidates) > 1 else None),
            candidate_top1=semantic["type"],
            confidence=semantic["confidence"],
            threshold_used=semantic["threshold"],
            margin=round(min(1.0, semantic["margin_score"] /
                             (semantic["score"] + 3.0)), 4),
            retrieval_mode="semantic_catalog",
            source="semantic",
            enrollable=False,
            topk=[(item["type"], item["score"]) for item in candidates],
            decision_reason=semantic["reason"],
            open_vocabulary=True,
            semantic_score=semantic["score"],
            semantic_evidence=semantic.get("evidence", []),
            catalog_version=semantic.get("catalog_version"),
        )

    def _apply_hierarchy(self, out):
        """Mâu thuẫn giữa các head -> hạ cả cụm xuống `unknown`.

        Đo ở notebook 04: precision 1.000 ở cả hai chế độ — mọi cửa sổ mâu thuẫn đều có
        ít nhất một head sai, và không cửa sổ đúng nào bị vứt nhầm.
        """
        if not self.hierarchy:
            for head in self.heads:
                out[head]["hierarchy_status"] = "not_configured"
            return
        model_result = out.get(self.model_head, {}) if self.model_head else {}
        if model_result.get("status") != "answer":
            for head in self.heads:
                out[head]["hierarchy_status"] = "no_model_answer"
            return
        rule = self.hierarchy.get(model_result.get("top1"))
        if rule is None:
            for head in self.heads:
                out[head]["hierarchy_status"] = "no_rule"
            return
        make, types = rule.get("make"), set(rule.get("types", []))
        make_result = out.get("make", {})
        type_result = out.get("type", {})
        bad_make = (make_result.get("status") == "answer"
                    and make_result.get("top1") != make)
        bad_type = (type_result.get("status") == "answer" and bool(types)
                    and type_result.get("top1") not in types)
        if bad_make or bad_type:
            for head in self.heads:
                out[head]["hierarchy_status"] = "conflict"
                if out[head]["status"] == "answer":
                    out[head].update(status="abstain", top1=None, conflict=True,
                                     decision_reason="hierarchy_conflict")
        else:
            for head in self.heads:
                out[head]["hierarchy_status"] = "consistent"

### `DeviceTracker` — gộp nhiều cửa sổ theo MAC

`DeviceTracker` - gom nhieu cua so cua cung mot MAC thanh MOT ket luan.

    t = DeviceTracker(predictor)
    for window in windows_of_this_mac:
        t.observe(window)
    print(t.status())          # identified / unknown / ambiguous / collecting

Day moi la cach dung dung ngoai hien truong; `Predictor` mot minh chi tra loi roi rac.

In [19]:
from collections import Counter


# Nguồn được ghim lại theo MAC. DHCP và TLS mang vân tay cấu trúc: thấy một lần là còn
# đúng mãi cho thiết bị đó. DNS/mDNS là nội dung theo thời điểm — thiết bị gọi về đâu
# thay đổi theo giờ, ghim lại sẽ bịa ra bằng chứng không có trong cửa sổ đang xét.
STICKY_SOURCES = ("dhcp", "tls")


# Mỗi trạng thái mức MAC ứng với một việc khác nhau cho người vận hành. Viết ra đây để
# tầng trên không phải tự suy, và để không ai gộp `ambiguous` với `unknown` lần nữa.
_REMEDY = {
    "identified": None,
    "collecting": "chờ thêm cửa sổ",
    "unknown": "thu nạp: xác nhận nhãn rồi thêm một dòng vào bảng L1",
    "unknown_no_fingerprint": "chờ MAC này lộ DHCP hoặc TLS — chưa có vân tay để thu nạp",
    "ambiguous": "vân tay đã biết nhưng không tách được nhãn; thu nạp vô ích, cần thêm nguồn bằng chứng",
    "unstable": "bằng chứng mâu thuẫn giữa các cửa sổ; xem lại có phải nhiều thiết bị dùng chung MAC không",
}


def bucket(result):
    """Kết quả một cửa sổ -> một trong ba rổ để đếm ở mức MAC.

    `ambiguous` chỉ được tính khi model CŨNG không trả lời được: vân tay nhập nhằng mà
    model vẫn đủ tự tin thì đó là một câu trả lời, không phải một ca cần người can thiệp.
    """
    if result["status"] == "answer":
        return "answer"
    return "ambiguous" if result.get("fp") == "ambiguous" else "unknown"


class DeviceTracker:
    """Gom nhiều cửa sổ của cùng một MAC thành một kết luận.

    Hai lý do bắt buộc phải quyết định ở mức MAC chứ không phải mức cửa sổ:

    1. **Bằng chứng mạnh nhất thì hiếm nhất.** DHCP chỉ xuất hiện ở ~10.7% số cửa sổ một
       giờ (kế hoạch §4.1b), nên 89% thời gian hệ thống mù về `dhcp_prl`/`dhcp_vci` nếu
       không ghim lại. Ghim theo MAC đưa vân tay đầy đủ vào mọi cửa sổ sau đó.
    2. **Một cửa sổ quá ồn để kết luận.** Đo ở notebook 04: luật "≥70% cửa sổ abstain"
       phát hiện 12/12 và 25/25 thiết bị lạ với 1/40 báo nhầm, và **3 cửa sổ là đủ** —
       kết quả ở N=3 và N=50 giống hệt nhau.
    """

    def __init__(self, predictor, min_windows=3, decide_ratio=0.7):
        self.p = predictor
        self.min_windows = min_windows
        self.decide_ratio = decide_ratio
        self.sticky = {}                 # nguồn -> {cột: giá trị} đã thấy
        self.windows = 0
        self.votes = {h: Counter() for h in predictor.heads}
        self.states = {h: Counter() for h in predictor.heads}
        self.last_row = None
        self.semantic_records = []

    def observe(self, records):
        """Nạp một cửa sổ. Trả kết quả của riêng cửa sổ đó."""
        row = self._apply_sticky(aggregate(records, self.p.feature_cols))
        self.last_row = row
        self._remember_semantic(records)
        out = self.p.predict_row(row, records=self.semantic_records)
        self.windows += 1
        for head in self.p.heads:
            r = out[head]
            self.states[head][bucket(r)] += 1
            if r["status"] == "answer":
                self.votes[head][r["top1"]] += 1
        return out

    def _remember_semantic(self, records, limit=128):
        """Persist only semantic protocol fields; never retain IP/MAC/payload metadata."""
        for record in records:
            if not isinstance(record, dict):
                continue
            proto = str(record.get("proto", "")).lower()
            fields = RECORD_FIELDS.get(proto)
            if not fields:
                continue
            compact = {"proto": proto}
            for field in fields:
                value = record.get(field)
                if value is not None and str(value).strip():
                    compact[field] = value
            if len(compact) > 1 and compact not in self.semantic_records:
                self.semantic_records.append(compact)
        if len(self.semantic_records) > limit:
            self.semantic_records = self.semantic_records[-limit:]

    def _apply_sticky(self, row):
        cols_by_source = {}
        for col in self.p.feature_cols:
            cols_by_source.setdefault(source_of(col), []).append(col)
        for src in STICKY_SOURCES:
            cols = cols_by_source.get(src, [])
            if not cols:
                continue
            if row.get(f"has_{src}") == 1:
                self.sticky[src] = {c: row[c] for c in cols}
            elif src in self.sticky:
                row.update(self.sticky[src])     # bằng chứng đã thấy ở cửa sổ trước
        return row

    def enroll_mode(self):
        """Thu nạp được ở mode nào với bằng chứng đang có. None = chưa đủ."""
        if self.last_row is None:
            return None
        mode, _ = self.p.table.best_mode(self.last_row)
        return mode

    def status(self):
        """Kết luận mức MAC."""
        mode = self.enroll_mode()
        out = {"windows": self.windows,
               "enroll_mode": mode,
               "enroll_provisional": mode == "fp_tls"}
        n = max(self.windows, 1)
        for head in self.p.heads:
            c = self.states[head]
            ratios = {k: c[k] / n for k in ("answer", "unknown", "ambiguous")}
            answered = c["answer"]
            top = self.votes[head].most_common(1)
            # Đồng thuận trong SỐ CỬA SỔ ĐÃ TRẢ LỜI, khác answer_ratio (tính trên TỔNG
            # cửa sổ). Cửa sổ 'ambiguous'/'unknown' không phát biểu nhãn nào nên không
            # tính là phản đối khi đo đồng thuận.
            agree_ratio = (top[0][1] / answered) if answered else 0.0
            if self.windows < self.min_windows:
                state = "collecting"
            elif ratios["answer"] >= self.decide_ratio:
                state = "identified"
            elif answered >= self.min_windows and agree_ratio >= self.decide_ratio:
                # Cửa sổ giàu bằng chứng (DHCP/mDNS) hiếm hơn cửa sổ nghèo (chỉ DNS/TLS)
                # — rõ nhất ở field-capture, nơi ngưỡng abstain ở n_sources thấp gần như
                # tuyệt đối nên phần lớn cửa sổ nghèo luôn abstain dù thiết bị đã biết.
                # ≥min_windows cửa sổ TRẢ LỜI và đồng thuận tuyệt đối đáng tin hơn tỉ lệ
                # trên tổng cửa sổ gợi ý, vì cửa sổ abstain không hề bất đồng — nó chỉ
                # không đủ bằng chứng để nói gì cả. Đo trên cả IoT Sentinel + CIC: không
                # đổi PHÁT HIỆN LẠ/BÁO NHẦM, không đổi SAI — chỉ chuyển một phần BỎ SÓT
                # thành OK.
                state = "identified"
            elif ratios["ambiguous"] >= self.decide_ratio:
                state = "ambiguous"
            elif ratios["unknown"] >= self.decide_ratio:
                # Lạ nhưng không có vân tay DHCP/TLS nào thì không thu nạp được: khoá
                # bảng L1 dựng từ hai nguồn đó, MAC chỉ nói DNS không tạo được khoá nào.
                state = "unknown" if mode else "unknown_no_fingerprint"
            else:
                state = "unstable"
            out[head] = {
                "state": state,
                "top1": top[0][0] if top else None,
                "votes": dict(self.votes[head]),
                "ratios": {k: round(v, 3) for k, v in ratios.items()},
                # Hai câu hỏi khác nhau, đừng gộp:
                #   needs_attention — "MAC này chưa nhận diện được" (gồm cả nhập nhằng)
                #   should_enroll   — "thêm một dòng vào bảng L1 sẽ giải quyết được"
                # Gộp hai cái làm một khiến thiết bị nhập nhằng biến mất khỏi tầm mắt
                # người vận hành, trong khi nó vẫn là thiết bị chưa nhận diện được.
                "needs_attention": state not in ("identified", "collecting"),
                "should_enroll": state == "unknown",
                "remedy": _REMEDY.get(state),
            }
        return out

    def enroll(self, labels, persist=True):
        """Người vận hành xác nhận nhãn cho MAC này -> ghi vào bảng thu nạp."""
        if self.last_row is None:
            raise RuntimeError("chưa quan sát cửa sổ nào")
        result = {head: self.p.table.enroll(self.last_row, head, label)
                  for head, label in labels.items()}
        if persist:
            write_enrolled(self.p.table)
        return result

## 4. Train và đánh giá open-set


### Đánh giá open-set

Reproducible open-set evaluation for the SDC classifier.

This module keeps the three validation populations separate:

* ``time``: train on earlier dates and test on later dates;
* ``unseen_device``: hold out a physical device while retaining only samples whose
  target label is still represented by another device in train;
* ``unseen_class``: remove one complete target class from train.

IoT Sentinel is intentionally not read here.  It remains an external test and must
not influence thresholds.

In [20]:
import argparse
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd



DEFAULT_TEST_FRACTION = 0.20
PREDICTION_COLUMNS = [
    "regime", "fold", "session_id", "device", "date", "n_sources",
    "source_combo", "head", "y_true", "label_in_train", "l1_pred",
    "l1_mode", "l2_pred", "l2_conf", "l2_p_of_l1",
]


def source_combinations(frame: pd.DataFrame) -> pd.Series:
    """Return a stable source-set label such as ``dhcp+dns+tls``."""
    values = []
    for row in frame[[f"has_{source}" for source in SOURCES]].itertuples(index=False):
        present = [source for source, flag in zip(SOURCES, row) if int(flag) == 1]
        values.append("+".join(present) if present else "none")
    return pd.Series(values, index=frame.index, dtype=object)


def chronological_split(
    frame: pd.DataFrame, test_fraction: float = DEFAULT_TEST_FRACTION
) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    """Split by a hard date boundary; no future date is allowed into train."""
    if not 0 < test_fraction < 1:
        raise ValueError("test_fraction must be between 0 and 1")
    dates = np.array(sorted(pd.to_datetime(frame["date"]).dt.normalize().unique()))
    if len(dates) < 2:
        raise ValueError("time evaluation needs at least two distinct dates")
    n_test = max(1, int(np.ceil(len(dates) * test_fraction)))
    n_test = min(n_test, len(dates) - 1)
    cutoff = pd.Timestamp(dates[-n_test])
    normalized = pd.to_datetime(frame["date"]).dt.normalize()
    train = frame.loc[normalized < cutoff].copy()
    test = frame.loc[normalized >= cutoff].copy()
    if train.empty or test.empty:
        raise ValueError("chronological split produced an empty partition")
    return train, test, cutoff.date().isoformat()


def _probability_of_label(classes, probabilities, labels) -> np.ndarray:
    positions = {str(label): i for i, label in enumerate(classes)}
    result = np.full(len(labels), np.nan, dtype=float)
    for i, label in enumerate(labels):
        if pd.notna(label) and str(label) in positions:
            result[i] = probabilities[i, positions[str(label)]]
    return result


def evaluate_fold(
    train: pd.DataFrame,
    test: pd.DataFrame,
    features: list[str],
    heads: list[str],
    regime: str,
    fold: str,
) -> pd.DataFrame:
    """Fit fold-local L1, encoder and classifiers, then predict every test row."""
    if train.empty or test.empty:
        raise ValueError("train and test must both contain rows")

    table = FingerprintTable().fit(train, heads)
    encoder = fit_encoder(train, features)
    x_train, _, _ = apply_encoder(train, encoder)
    x_test, _, _ = apply_encoder(test, encoder)
    source_combo = source_combinations(test)
    session_ids = test.index.astype(str).to_numpy()
    devices = test["canonical_device"].astype(str).to_numpy()
    dates = test["date"].astype(str).to_numpy()
    n_sources = test["n_sources"].astype(int).to_numpy()
    combos = source_combo.to_numpy()
    output = []

    for head in heads:
        model = make_model().fit(x_train, train[head].astype(str).to_numpy())
        probabilities = model.predict_proba(x_test)
        best_index = probabilities.argmax(axis=1)
        l2_pred = model.classes_[best_index].astype(str)
        l2_conf = probabilities[np.arange(len(test)), best_index]
        l1_pred, l1_mode = table.predict(test, head)
        l2_p_of_l1 = _probability_of_label(model.classes_, probabilities, l1_pred)
        seen = set(train[head].astype(str))

        part = pd.DataFrame({
            "regime": regime,
            "fold": str(fold),
            "session_id": session_ids,
            "device": devices,
            "date": dates,
            "n_sources": n_sources,
            "source_combo": combos,
            "head": head,
            "y_true": test[head].astype(str).to_numpy(),
            "label_in_train": test[head].astype(str).isin(seen).to_numpy(),
            "l1_pred": l1_pred.to_numpy(),
            "l1_mode": l1_mode.to_numpy(),
            "l2_pred": l2_pred,
            "l2_conf": l2_conf,
            "l2_p_of_l1": l2_p_of_l1,
        })
        output.append(part)

    return pd.concat(output, ignore_index=True)[PREDICTION_COLUMNS]


def evaluate_time(
    frame: pd.DataFrame, features: list[str], test_fraction: float
) -> tuple[pd.DataFrame, dict]:
    train, test, cutoff = chronological_split(frame, test_fraction)
    predictions = evaluate_fold(
        train, test, features, list(LABEL_COLS), "time", cutoff
    )
    metadata = {
        "cutoff": cutoff,
        "train_rows": int(len(train)),
        "test_rows": int(len(test)),
        "train_dates": int(train["date"].nunique()),
        "test_dates": int(test["date"].nunique()),
    }
    return predictions, metadata


def evaluate_unseen_devices(frame: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Run true leave-one-device-out. This is slower than reusing a run's OOF file."""
    output = []
    for device in sorted(frame["canonical_device"].astype(str).unique()):
        held_out = frame["canonical_device"].astype(str).eq(device)
        output.append(evaluate_fold(
            frame.loc[~held_out], frame.loc[held_out], features,
            list(LABEL_COLS), "unseen_device", device,
        ))
    return pd.concat(output, ignore_index=True)


def evaluate_unseen_classes(frame: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Remove every complete class in turn; each fold refits vocab and L1 from train only."""
    output = []
    for head in LABEL_COLS:
        labels = sorted(frame[head].astype(str).unique())
        for number, label in enumerate(labels, start=1):
            print(f"unseen_class {head}: {number}/{len(labels)} - {label}", flush=True)
            held_out = frame[head].astype(str).eq(label)
            output.append(evaluate_fold(
                frame.loc[~held_out], frame.loc[held_out], features,
                [head], "unseen_class", label,
            ))
    return pd.concat(output, ignore_index=True)


def normalize_cached_oof(
    path: Path, frame: pd.DataFrame, regime: str = "ood_device"
) -> pd.DataFrame:
    """Upgrade one cached OOF regime to the evaluator schema."""
    cached = pd.read_parquet(path)
    cached = cached[cached["regime"].eq(regime)].copy()
    if cached.empty:
        raise ValueError(f"cached OOF has no {regime!r} rows")
    required = {
        "regime", "fold", "session_id", "device", "n_sources", "head",
        "y_true", "label_in_train", "l1_pred", "l1_mode", "l2_pred",
        "l2_conf", "l2_p_of_l1",
    }
    missing = required - set(cached.columns)
    if missing:
        raise ValueError(f"cached OOF is missing columns: {sorted(missing)}")
    lookup = frame.copy()
    lookup["session_id"] = lookup.index.astype(str)
    lookup["source_combo"] = source_combinations(lookup)
    lookup = lookup.set_index("session_id")
    cached["session_id"] = cached["session_id"].astype(str)
    cached["date"] = cached["session_id"].map(lookup["date"].astype(str))
    cached["source_combo"] = cached["session_id"].map(lookup["source_combo"])
    if cached[["date", "source_combo"]].isna().any().any():
        raise ValueError("cached OOF contains sessions not found in sessions.parquet")
    cached["fold"] = cached["fold"].astype(str)
    return cached[PREDICTION_COLUMNS]


def cached_oof_has_regime(path: Path, regime: str) -> bool:
    if not path.exists():
        return False
    regimes = pd.read_parquet(path, columns=["regime"])["regime"]
    return bool(regimes.eq(regime).any())

In [21]:
def apply_policy(predictions: pd.DataFrame, bundle: dict) -> pd.DataFrame:
    """Apply the tiered decision policy exactly, before hierarchy checking."""
    if bundle.get("format") != TIERED_FORMAT:
        raise ValueError("open-set evaluator requires an sdc-tiered-v2 bundle")
    thresholds, min_sources, l1_floor = validate_tiered_policy(
        bundle, LABEL_COLS
    )
    combo_thresholds = validate_source_combo_thresholds(bundle, LABEL_COLS)
    result = predictions.copy()
    result["answered"] = False
    result["prediction"] = None
    result["decision_path"] = "abstain"
    result["confidence"] = result["l2_conf"].astype(float)
    result["threshold_used"] = np.nan

    l1_hit = result["l1_pred"].notna()
    result.loc[l1_hit, "threshold_used"] = l1_floor
    l1_answer = l1_hit & result["l2_p_of_l1"].ge(l1_floor)
    result.loc[l1_answer, "answered"] = True
    result.loc[l1_answer, "prediction"] = result.loc[l1_answer, "l1_pred"]
    result.loc[l1_answer, "decision_path"] = "l1"
    result.loc[l1_hit, "confidence"] = result.loc[l1_hit, "l2_p_of_l1"]

    l2_candidate = ~l1_hit
    for head in LABEL_COLS:
        for count in range(1, len(SOURCES) + 1):
            selected = l2_candidate & result["head"].eq(head) & result["n_sources"].eq(count)
            if not selected.any():
                continue
            default_threshold = thresholds[f"{head}|{count}"]
            row_thresholds = result.loc[selected, "source_combo"].map(
                lambda combo: combo_thresholds.get(f"{head}|{combo}", default_threshold)
            )
            result.loc[selected, "threshold_used"] = row_thresholds
            answer = selected & (count >= min_sources[head]) & result["l2_conf"].ge(
                result["threshold_used"]
            )
            result.loc[answer, "answered"] = True
            result.loc[answer, "prediction"] = result.loc[answer, "l2_pred"]
            result.loc[answer, "decision_path"] = "l2"

    result["correct"] = result["answered"] & result["prediction"].eq(result["y_true"])
    return result


def apply_hierarchy(predictions: pd.DataFrame, hierarchy: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply model/make/type consistency to regimes where all heads share a fold."""
    result = predictions.copy()
    result["pre_hierarchy_answered"] = result["answered"]
    result["pre_hierarchy_prediction"] = result["prediction"]
    result["pre_hierarchy_correct"] = result["correct"]
    result["hierarchy_conflict"] = False
    detail = []
    eligible = result[result["regime"].ne("unseen_class")]
    keys = ["regime", "fold", "session_id"]
    conflict_keys = []
    for key, group in eligible.groupby(keys, sort=False):
        by_head = {row.head: row for row in group.itertuples()}
        # Chấp nhận cả tên head cũ để đọc lại được OOF của những run trước khi đổi tên.
        model_row = by_head.get(MODEL_HEAD) or by_head.get(LEGACY_MODEL_HEAD)
        if model_row is None or not model_row.answered:
            continue
        rule = hierarchy.get(model_row.prediction)
        if not rule:
            continue
        make = by_head.get("make")
        kind = by_head.get("type")
        bad_make = make is not None and make.answered and make.prediction != rule.get("make")
        types = set(rule.get("types", []))
        bad_type = kind is not None and kind.answered and types and kind.prediction not in types
        if not (bad_make or bad_type):
            continue
        conflict_keys.append(key)
        detail.append({
            "regime": key[0], "fold": key[1], "session_id": key[2],
            "bad_make": bool(bad_make), "bad_type": bool(bad_type),
        })
    if conflict_keys:
        row_keys = pd.MultiIndex.from_frame(result[keys])
        conflict_index = pd.MultiIndex.from_tuples(conflict_keys, names=keys)
        mask = row_keys.isin(conflict_index)
        answered = mask & result["answered"].to_numpy(dtype=bool)
        result.loc[mask, "hierarchy_conflict"] = True
        result.loc[answered, "answered"] = False
        result.loc[answered, "prediction"] = None
        result.loc[answered, "correct"] = False
        result.loc[answered, "decision_path"] = "hierarchy_abstain"
    return result, pd.DataFrame(
        detail,
        columns=["regime", "fold", "session_id", "bad_make", "bad_type"],
    )


def _metric_row(group: pd.DataFrame, regime: str, head: str, population: str,
                level: str, breakdown: str = "all", value: str = "all") -> dict:
    n = len(group)
    answered = group["answered"].astype(bool)
    n_answered = int(answered.sum())
    return {
        "regime": regime,
        "head": head,
        "population": population,
        "level": level,
        "breakdown": breakdown,
        "value": value,
        "n": int(n),
        "n_answered": n_answered,
        "coverage": n_answered / n if n else np.nan,
        "accuracy_answered": (float(group.loc[answered, "correct"].mean())
                              if n_answered else np.nan),
        "false_positive_rate": (n_answered / n if population == "unknown" and n else np.nan),
    }


def window_metrics(predictions: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (regime, head), base in predictions.groupby(["regime", "head"], sort=True):
        populations = {
            "all": base,
            "known": base[base["label_in_train"]],
            "unknown": base[~base["label_in_train"]],
        }
        for population, group in populations.items():
            if group.empty:
                continue
            rows.append(_metric_row(group, regime, head, population, "window"))
            for count, part in group.groupby("n_sources"):
                rows.append(_metric_row(
                    part, regime, head, population, "window", "n_sources", str(count)
                ))
            for combo, part in group.groupby("source_combo"):
                rows.append(_metric_row(
                    part, regime, head, population, "window", "source_combo", str(combo)
                ))
    return pd.DataFrame(rows)


def device_metrics(predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    device_rows = []
    for (regime, head, device), group in predictions.groupby(
        ["regime", "head", "device"], sort=True
    ):
        answers = group[group["answered"]]
        prediction = None
        if not answers.empty:
            counts = answers["prediction"].value_counts()
            prediction = sorted(counts[counts.eq(counts.max())].index.astype(str))[0]
        truth = str(group["y_true"].mode().iloc[0])
        device_rows.append({
            "regime": regime,
            "head": head,
            "device": device,
            "label_in_train": bool(group["label_in_train"].all()),
            "y_true": truth,
            "prediction": prediction,
            "answered": prediction is not None,
            "correct": prediction == truth,
            "n_windows": int(len(group)),
            "n_answered_windows": int(len(answers)),
            "window_coverage": len(answers) / len(group),
        })
    detail = pd.DataFrame(device_rows)
    rows = []
    for (regime, head), base in detail.groupby(["regime", "head"], sort=True):
        for population, group in {
            "all": base,
            "known": base[base["label_in_train"]],
            "unknown": base[~base["label_in_train"]],
        }.items():
            if group.empty:
                continue
            rows.append(_metric_row(group, regime, head, population, "device"))
    return detail, pd.DataFrame(rows)


def calibration_table(predictions: pd.DataFrame) -> pd.DataFrame:
    known = predictions[predictions["label_in_train"] & predictions["l1_pred"].isna()].copy()
    if known.empty:
        return pd.DataFrame()
    known["l2_correct"] = known["l2_pred"].eq(known["y_true"])
    known["bin"] = pd.cut(
        known["l2_conf"], bins=np.linspace(0, 1, 11), include_lowest=True
    ).astype(str)
    return (known.groupby(["regime", "head", "bin"], observed=True)
            .agg(n=("l2_conf", "size"), mean_confidence=("l2_conf", "mean"),
                 empirical_accuracy=("l2_correct", "mean"))
            .reset_index())


def write_confusions(predictions: pd.DataFrame, output_dir: Path) -> list[str]:
    files = []
    answered = predictions[predictions["answered"] & predictions["label_in_train"]]
    for (regime, head), group in answered.groupby(["regime", "head"], sort=True):
        table = pd.crosstab(group["y_true"], group["prediction"], dropna=False)
        path = output_dir / f"confusion_{regime}_{head}.csv"
        table.to_csv(path)
        files.append(path.name)
    return files

In [22]:
def evaluate_parse_args(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--run", help="Model run directory or run id; defaults to current_model.json")
    parser.add_argument("--test-fraction", type=float, default=DEFAULT_TEST_FRACTION)
    parser.add_argument("--recompute-unseen-device", action="store_true",
                        help="Recompute LOGO instead of using the run's cached oof_pred.parquet")
    parser.add_argument("--recompute-unseen-class", action="store_true",
                        help="Recompute class holdouts instead of using cached OOF rows")
    parser.add_argument("--skip-unseen-class", action="store_true",
                        help="Useful only for a quick smoke run")
    parser.add_argument("--output-dir", type=Path)
    return parser.parse_args(argv)


def evaluate_main(argv=None) -> Path:
    args = evaluate_parse_args(argv)
    run_dir = resolve_run_dir(args.run)
    bundle = joblib.load(run_dir / "model.joblib")
    frame, feature_sets = load_sessions()
    features = feature_sets["all"]
    output_dir = args.output_dir or (REPORTS / f"open_set_{run_dir.name}")
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"run={run_dir.name}; sessions={len(frame)}; features={len(features)}", flush=True)
    time_predictions, time_meta = evaluate_time(frame, features, args.test_fraction)

    cached_oof = run_dir / "oof_pred.parquet"
    if cached_oof_has_regime(cached_oof, "ood_device") and not args.recompute_unseen_device:
        print(f"reuse {cached_oof}", flush=True)
        device_predictions = normalize_cached_oof(cached_oof, frame, "ood_device")
    else:
        device_predictions = evaluate_unseen_devices(frame, features)

    parts = [time_predictions, device_predictions]
    if not args.skip_unseen_class:
        if cached_oof_has_regime(cached_oof, "unseen_class") and not args.recompute_unseen_class:
            print(f"reuse unseen_class from {cached_oof}", flush=True)
            parts.append(normalize_cached_oof(cached_oof, frame, "unseen_class"))
        else:
            parts.append(evaluate_unseen_classes(frame, features))
    raw = pd.concat(parts, ignore_index=True)
    decided = apply_policy(raw, bundle)
    decided, conflicts = apply_hierarchy(decided, bundle.get("hierarchy", {}))

    window = window_metrics(decided)
    device_detail, device_summary = device_metrics(decided)
    metrics = pd.concat([window, device_summary], ignore_index=True)
    calibration = calibration_table(decided)

    raw.to_parquet(output_dir / "raw_predictions.parquet", index=False)
    decided.to_parquet(output_dir / "decisions.parquet", index=False)
    metrics.to_csv(output_dir / "metrics.csv", index=False)
    device_detail.to_csv(output_dir / "device_metrics.csv", index=False)
    calibration.to_csv(output_dir / "calibration.csv", index=False)
    conflicts.to_csv(output_dir / "hierarchy_conflicts.csv", index=False)
    confusion_files = write_confusions(decided, output_dir)

    headline = metrics[
        metrics["breakdown"].eq("all") & metrics["level"].isin(["window", "device"])
    ]
    summary = {
        "format": "sdc-open-set-evaluation-v1",
        "run_id": run_dir.name,
        "data": str(SESSIONS_PATH),
        "n_sessions": int(len(frame)),
        "n_devices": int(frame["canonical_device"].nunique()),
        "time_split": time_meta,
        "external_test_used_for_tuning": False,
        "warnings": [
            "Thresholds are calibrated only on internal CIC folds; IoT Sentinel is external test only.",
            "Open-vocabulary semantic type rules are evaluated separately by Code/06_test_model.ipynb.",
        ],
        "headline": headline.replace({np.nan: None}).to_dict(orient="records"),
        "hierarchy_conflicts": int(len(conflicts)),
        "confusion_files": confusion_files,
    }
    (output_dir / "summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"wrote {output_dir}", flush=True)
    return output_dir

### Train và hiệu chỉnh policy

Train a reproducible SDC tiered-v2 bundle from verified sessions.

Threshold calibration uses only CIC internal folds.  The IoT Sentinel external set is
never read by this script.  The resulting run is not activated automatically; callers
must validate it and then update ``Models/current_model.json`` explicitly.

In [23]:
import argparse
from datetime import datetime
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold



TARGET_ACCURACY = {"make": 0.99, "type": 0.90, MODEL_HEAD: 0.99}

# Ô ngưỡng do người vận hành ấn định, ghi đè lựa chọn của hàm mục tiêu.
#
# `type|4`: hàm mục tiêu `coverage + ood_abstain - 1` chọn 0.48 (coverage 1.000,
# ood_abstain 0.714, accuracy 0.958) vì nó cân coverage với chống lỗi im lặng NGANG
# nhau. Mục tiêu của sản phẩm là open-set, nên đánh đổi đó sai chiều: ở 0.80 đường cong
# OOF cho ood_abstain 1.000 và accuracy 1.000, đổi lấy coverage 0.667. Chênh lệch
# objective chỉ 0.047, còn chênh lệch về lỗi im lặng là toàn bộ 28.6% còn lại.
#
# Ghi đè chứ không sửa hàm mục tiêu: hàm mục tiêu đúng cho `make`/`model`, và một ô
# ấn định bằng tay thì phải nhìn thấy được ở đây chứ không giấu trong công thức.
THRESHOLD_OVERRIDES = {"type|4": 0.80}
DEFAULT_MIN_SOURCES = {head: 2 for head in LABEL_COLS}
THRESHOLD_GRID = np.linspace(0.0, 1.0, 201)
MIN_L1_FLOOR_WITHOUT_OOD = 0.50


def validate_training_frame(frame: pd.DataFrame) -> None:
    required = {"canonical_device", "date", "n_sources", *LABEL_COLS}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"training dataset is missing columns: {sorted(missing)}")
    if frame[list(LABEL_COLS)].isna().any().any():
        raise ValueError("training labels contain missing values")
    if frame["make"].astype(str).str.casefold().eq("unknown").any():
        raise ValueError("make=Unknown is forbidden in a production model")
    model_make = frame.groupby(MODEL_HEAD)["make"].nunique()
    model_type = frame.groupby(MODEL_HEAD)["type"].nunique()
    if model_make.ne(1).any() or model_type.ne(1).any():
        raise ValueError("verified model must map to exactly one make and one type")


def build_hierarchy(frame: pd.DataFrame) -> dict:
    unique = frame[[MODEL_HEAD, "make", "type"]].drop_duplicates()
    return {
        str(name): {
            "make": str(group["make"].iloc[0]),
            "types": sorted(group["type"].astype(str).unique()),
        }
        for name, group in unique.groupby(MODEL_HEAD, sort=True)
    }


def make_date_oof(frame: pd.DataFrame, features: list[str], n_splits: int) -> pd.DataFrame:
    groups = date_groups(frame)
    n_splits = min(n_splits, int(pd.Series(groups).nunique()))
    if n_splits < 2:
        raise ValueError("date OOF needs at least two distinct groups")
    output = []
    splitter = GroupKFold(n_splits=n_splits)
    for fold, (train_index, test_index) in enumerate(
        splitter.split(frame, groups=groups), start=1
    ):
        print(f"id_date fold {fold}/{n_splits}", flush=True)
        output.append(evaluate_fold(
            frame.iloc[train_index], frame.iloc[test_index], features,
            list(LABEL_COLS), "id_date", str(fold),
        ))
    return pd.concat(output, ignore_index=True)


def _curve_for_group(id_group: pd.DataFrame, ood_group: pd.DataFrame, target: float) -> pd.DataFrame:
    rows = []
    for threshold in THRESHOLD_GRID:
        id_answer = id_group["l2_conf"].ge(threshold)
        ood_answer = ood_group["l2_conf"].ge(threshold)
        n_answered = int(id_answer.sum())
        accuracy = (
            float(id_group.loc[id_answer, "l2_pred"].eq(
                id_group.loc[id_answer, "y_true"]
            ).mean()) if n_answered else np.nan
        )
        coverage = float(id_answer.mean()) if len(id_group) else 0.0
        ood_abstain = float((~ood_answer).mean()) if len(ood_group) else 1.0
        rows.append({
            "threshold": float(threshold),
            "id_coverage": coverage,
            "id_accuracy_answered": accuracy,
            "ood_abstain": ood_abstain,
            "n_id": int(len(id_group)),
            "n_ood": int(len(ood_group)),
            "target_accuracy": target,
            "target_reached": bool(n_answered and accuracy >= target),
            "objective": coverage + ood_abstain - 1.0,
        })
    return pd.DataFrame(rows)


def choose_threshold(curve: pd.DataFrame) -> pd.Series:
    feasible = curve[curve["target_reached"]]
    if not feasible.empty:
        # Prefer the open-set objective, then coverage, then the stricter threshold.
        return feasible.sort_values(
            ["objective", "id_coverage", "threshold"], ascending=[False, False, False]
        ).iloc[0]
    answered = curve[curve["id_accuracy_answered"].notna()]
    if answered.empty:
        return curve.iloc[-1]
    return answered.sort_values(
        ["id_accuracy_answered", "ood_abstain", "id_coverage", "threshold"],
        ascending=[False, False, False, False],
    ).iloc[0]


def calibrate_l2(id_oof: pd.DataFrame, class_oof: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    thresholds = {}
    selected = []
    # L1 decisions use a separate backing floor and must not influence L2 thresholds.
    id_l2 = id_oof[id_oof["l1_pred"].isna() & id_oof["label_in_train"]]
    ood_l2 = class_oof[class_oof["l1_pred"].isna() & ~class_oof["label_in_train"]]
    for head in LABEL_COLS:
        for count in range(1, len(SOURCES) + 1):
            known = id_l2[id_l2["head"].eq(head) & id_l2["n_sources"].eq(count)]
            unknown = ood_l2[ood_l2["head"].eq(head) & ood_l2["n_sources"].eq(count)]
            curve = _curve_for_group(known, unknown, TARGET_ACCURACY[head])
            key = f"{head}|{count}"
            if key in THRESHOLD_OVERRIDES:
                # Lấy đúng dòng trên đường cong để coverage/accuracy/ood_abstain trong
                # `thresholds.csv` khớp ngưỡng thật sự dùng, không phải ngưỡng bị bỏ.
                wanted = float(THRESHOLD_OVERRIDES[key])
                choice = curve.loc[(curve["threshold"] - wanted).abs().idxmin()].to_dict()
                choice["overridden"] = True
            else:
                choice = choose_threshold(curve).to_dict()
                choice["overridden"] = False
            thresholds[key] = round(float(choice["threshold"]), 4)
            choice.update({"head": head, "n_sources": count})
            selected.append(choice)
    return thresholds, pd.DataFrame(selected)


def calibrate_l1_floor(id_oof: pd.DataFrame, class_oof: pd.DataFrame) -> tuple[float, dict]:
    known = id_oof[id_oof["l1_pred"].notna() & id_oof["label_in_train"]].copy()
    unknown = class_oof[class_oof["l1_pred"].notna() & ~class_oof["label_in_train"]].copy()
    if known.empty:
        return 1.0, {"n_id": 0, "n_ood": int(len(unknown)), "reason": "no fold-local L1 hits"}
    known["l1_correct"] = known["l1_pred"].eq(known["y_true"])
    rows = []
    for threshold in THRESHOLD_GRID:
        selected = known["l2_p_of_l1"].ge(threshold)
        per_head_ok = True
        for head in LABEL_COLS:
            head_selected = selected & known["head"].eq(head)
            if head_selected.any():
                per_head_ok &= bool(
                    known.loc[head_selected, "l1_correct"].mean() >= TARGET_ACCURACY[head]
                )
        ood_answer = unknown["l2_p_of_l1"].ge(threshold)
        coverage = float(selected.mean())
        abstain = float((~ood_answer).mean()) if len(unknown) else 1.0
        rows.append({
            "threshold": float(threshold), "target_reached": bool(per_head_ok and selected.any()),
            "id_coverage": coverage, "ood_abstain": abstain,
            "objective": coverage + abstain - 1.0,
        })
    curve = pd.DataFrame(rows)
    choice = choose_threshold(curve.assign(
        id_accuracy_answered=np.where(curve.target_reached, 1.0, np.nan)
    ))
    floor = round(float(choice["threshold"]), 4)
    conservative_fallback = unknown.empty
    if conservative_fallback:
        floor = max(floor, MIN_L1_FLOOR_WITHOUT_OOD)
    return floor, {
        "threshold": floor,
        "n_id": int(len(known)),
        "n_ood": int(len(unknown)),
        "id_coverage": float(choice["id_coverage"]),
        "ood_abstain": float(choice["ood_abstain"]),
        "target_reached": bool(choice["target_reached"]),
        "conservative_fallback": conservative_fallback,
    }


def train_final(frame: pd.DataFrame, features: list[str], policy: dict) -> dict:
    encoder = fit_encoder(frame, features)
    x_all, feature_names, _ = apply_encoder(frame, encoder)
    models = {}
    for head in LABEL_COLS:
        print(f"final model: {head}", flush=True)
        models[head] = make_model().fit(x_all, frame[head].astype(str))
    table = FingerprintTable().fit(frame, LABEL_COLS)
    return {
        "format": TIERED_FORMAT,
        "heads": list(LABEL_COLS),
        "feature_cols": features,
        "feature_names": feature_names,
        "encoder": encoder,
        "models": models,
        "l1_tables": dict(table.tables),
        "l1_ambiguous": dict(table.ambiguous),
        "hierarchy": build_hierarchy(frame),
        **policy,
    }


def train_parse_args(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--sessions", type=Path, default=SESSIONS_PATH)
    parser.add_argument("--date-folds", type=int, default=5)
    parser.add_argument("--skip-device-oof", action="store_true",
                        help="Faster development run; production runs should retain LOGO OOF")
    parser.add_argument("--run-id")
    parser.add_argument("--reuse-oof-from", type=Path,
                        help="Reuse compatible OOF parquet from a previous candidate run")
    return parser.parse_args(argv)

In [24]:
def train_main(argv=None) -> Path:
    args = train_parse_args(argv)
    frame, feature_sets = load_sessions(args.sessions)
    validate_training_frame(frame)
    features = feature_sets["all"]
    run_id = args.run_id or datetime.now().strftime("%Y%m%d_%H%M%S_verified_tiered")
    run_dir = MODELS / run_id
    if run_dir.exists():
        raise FileExistsError(f"run already exists: {run_dir}")
    run_dir.mkdir(parents=True)

    print(
        f"training {run_id}: {len(frame)} sessions, "
        f"{frame.canonical_device.nunique()} devices, {len(features)} features",
        flush=True,
    )
    reused_oof = None
    if args.reuse_oof_from:
        reuse_path = args.reuse_oof_from
        if reuse_path.is_dir():
            reuse_path = reuse_path / "oof_pred.parquet"
        reused_oof = pd.read_parquet(reuse_path)
        expected = {"id_date", "unseen_class", "unseen_device"}
        missing = expected - set(reused_oof["regime"].unique())
        if missing:
            raise ValueError(f"reused OOF is missing regimes: {sorted(missing)}")
        session_ids = set(frame.index.astype(str))
        if not set(reused_oof["session_id"].astype(str)).issubset(session_ids):
            raise ValueError("reused OOF contains sessions absent from the training dataset")
        print(f"reuse OOF from {reuse_path}", flush=True)
        id_oof = reused_oof[reused_oof["regime"].eq("id_date")].copy()
        class_oof = reused_oof[reused_oof["regime"].eq("unseen_class")].copy()
    else:
        id_oof = make_date_oof(frame, features, args.date_folds)
        class_oof = evaluate_unseen_classes(frame, features)
    thresholds, calibration = calibrate_l2(id_oof, class_oof)
    l1_floor, l1_calibration = calibrate_l1_floor(id_oof, class_oof)
    policy = {
        "thresholds_by_source": thresholds,
        "min_sources": dict(DEFAULT_MIN_SOURCES),
        "l1_floor": l1_floor,
    }
    # Validate before serializing so a malformed policy can never become a run artifact.
    validate_tiered_policy({"format": TIERED_FORMAT, **policy}, LABEL_COLS)
    bundle = train_final(frame, features, policy)

    oof_parts = [id_oof]
    if not args.skip_device_oof:
        if reused_oof is not None:
            oof_parts.append(reused_oof[reused_oof["regime"].eq("unseen_device")].copy())
        else:
            oof_parts.append(evaluate_unseen_devices(frame, features))
    oof_parts.append(class_oof)
    oof = pd.concat(oof_parts, ignore_index=True)
    joblib.dump(bundle, run_dir / "model.joblib", compress=3)
    oof.to_parquet(run_dir / "oof_pred.parquet", index=False)
    calibration.to_csv(run_dir / "thresholds.csv", index=False)
    metadata = {
        "format": TIERED_FORMAT,
        "run_id": run_id,
        "created": datetime.now().isoformat(timespec="seconds"),
        "dataset": str(args.sessions.resolve()),
        "taxonomy": str((DATA / "device_labels_verified.csv").resolve()),
        "external_test_used_for_tuning": False,
        "n_sessions": int(len(frame)),
        "n_devices": int(frame.canonical_device.nunique()),
        "n_estimators": int(N_ESTIMATORS),
        "heads": {head: int(frame[head].nunique()) for head in LABEL_COLS},
        "thresholds_by_source": thresholds,
        "min_sources": dict(DEFAULT_MIN_SOURCES),
        "l1_calibration": l1_calibration,
        "oof_regimes": sorted(oof.regime.unique()),
        "oof_reused_from": str(args.reuse_oof_from) if args.reuse_oof_from else None,
    }
    (run_dir / "meta.json").write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"wrote candidate run {run_dir}; manifest unchanged", flush=True)
    return run_dir

In [25]:
def select_run_dir(run=None):
    """Select an explicit run, the pinned run, or the only available candidate."""
    if run is not None:
        return resolve_run_dir(run)
    if CURRENT_MODEL_MANIFEST.is_file():
        return resolve_run_dir()
    candidates = sorted(p for p in MODELS.iterdir() if (p / "model.joblib").is_file()) if MODELS.exists() else []
    if len(candidates) != 1:
        raise RuntimeError(
            "Hãy đặt RUN_DIR cụ thể: chưa có model ghim hoặc có nhiều candidate trong Models/."
        )
    return candidates[0]


SDC_DEFS_LOADED = True
print("Đã nạp hàm SDC từ 03_train_model.ipynb")

Đã nạp hàm SDC từ 03_train_model.ipynb


## 5. Tạo candidate

Đầu vào là `Data/sessions_verified.parquet` từ notebook `02`. Chỉ bật `REBUILD_VERIFIED` nếu cần dựng lại bản đã xác minh từ `Data/sessions.parquet` và taxonomy. Train ghi artifact vào `Models/<run_id>/`; chưa kích hoạt model phục vụ.


In [26]:
RUN_ID = None
DATE_FOLDS = 5
SKIP_DEVICE_OOF = False
REUSE_OOF_FROM = None
REBUILD_VERIFIED = False


In [27]:
if not globals().get("SDC_IMPORT_ONLY", False):
    from IPython.display import display

    if REBUILD_VERIFIED:
        build_verified_dataset(
            source=ROOT / "Data" / "sessions.parquet",
            labels_path=ROOT / "Data" / "device_labels_verified.csv",
            excluded_path=ROOT / "Data" / "device_labels_excluded.csv",
            output=SESSIONS_PATH,
        )
    sessions, feature_groups = load_sessions(SESSIONS_PATH)
    print(f"Dataset: {len(sessions)} session, {sessions.canonical_device.nunique()} thiết bị")

    import warnings
    warnings.filterwarnings("ignore", category=UserWarning, module=r"sklearn\.utils\.parallel")

    train_args = ["--sessions", str(SESSIONS_PATH), "--date-folds", str(DATE_FOLDS)]
    if RUN_ID:
        train_args += ["--run-id", RUN_ID]
    if SKIP_DEVICE_OOF:
        train_args += ["--skip-device-oof"]
    if REUSE_OOF_FROM:
        train_args += ["--reuse-oof-from", str(Path(REUSE_OOF_FROM))]
    run_dir = train_main(train_args).resolve()
    predictor = Predictor(run_dir, enrolled={})
    print("Candidate:", run_dir)
    print("Contract:", predictor.contract_format)
    display(pd.read_csv(run_dir / "thresholds.csv"))


Dataset: 8189 session, 51 thiết bị
training 20260915_132703_verified_tiered: 8189 sessions, 51 devices, 40 features
id_date fold 1/5
id_date fold 2/5
id_date fold 3/5
id_date fold 4/5
id_date fold 5/5
unseen_class make: 1/26 - Amazon
unseen_class make: 2/26 - Amcrest
unseen_class make: 3/26 - Apple
unseen_class make: 4/26 - Arlo
unseen_class make: 5/26 - Borun
unseen_class make: 6/26 - Camera
unseen_class make: 7/26 - D-Link
unseen_class make: 8/26 - Eufy
unseen_class make: 9/26 - Generic Laptop
unseen_class make: 10/26 - Google
unseen_class make: 11/26 - HeimVision
unseen_class make: 12/26 - Home Eye
unseen_class make: 13/26 - LG
unseen_class make: 14/26 - Linova/Linux
unseen_class make: 15/26 - Luohe
unseen_class make: 16/26 - Netatmo
unseen_class make: 17/26 - OPPO
unseen_class make: 18/26 - Philips
unseen_class make: 19/26 - Raspberry Pi
unseen_class make: 20/26 - Ring
unseen_class make: 21/26 - Samsung
unseen_class make: 22/26 - SimCam
unseen_class make: 23/26 - Sonos
unseen_class

,threshold,id_coverage,id_accuracy_answered,ood_abstain,n_id,n_ood,target_accuracy,target_reached,objective,overridden,head,n_sources
0,1.00,0.987642,1.0,0.925561,2023,2230,0.99,True,0.913203,False,make,1
1,0.96,0.985370,1.0,0.999772,2529,4392,0.99,True,0.985142,False,make,2
2,0.92,0.963883,1.0,0.995839,886,1442,0.99,True,0.959722,False,make,3
3,0.48,0.968750,1.0,0.976000,96,125,0.99,True,0.944750,False,make,4
4,1.00,0.985707,1.0,1.000000,2029,2229,0.90,True,0.985707,False,type,1
5,1.00,0.960652,1.0,1.000000,2516,4392,0.90,True,0.960652,False,type,2
6,0.72,0.986456,1.0,0.947989,886,1442,0.90,True,0.934445,False,type,3
7,0.80,0.791667,1.0,1.000000,96,125,0.90,True,0.791667,True,type,4
8,1.00,0.985171,1.0,0.925561,2023,2230,0.99,True,0.910731,False,model,1
9,1.00,0.959272,1.0,1.000000,2529,4392,0.99,True,0.959272,False,model,2
